#1. Environment Setup

In [ ]:
import os
import io
import requests
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.compute as pc
from datetime import datetime
from dateutil.relativedelta import relativedelta
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from sklearn.metrics import mean_absolute_error, mean_squared_error
from io import StringIO
from plotly.subplots import make_subplots
from scipy import stats
from scipy.stats import pearsonr
from collections import Counter
from statsmodels.graphics.tsaplots import acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf as sm_plot_pacf
import shap
import lightgbm as lgb
import logging
import statsmodels.api as sm
import plotly.graph_objects as go
from prophet import Prophet
import xml.etree.ElementTree as ET
import pandas as pd
import requests
from io import StringIO
from matplotlib.colors import LinearSegmentedColormap
import math

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = "/content/drive/MyDrive/MSc IBS/Capstone Project"
    print(f"Colab — BASE: {BASE}")
except ImportError:
    BASE = os.path.abspath(os.path.join(os.path.dirname(os.path.abspath("__file__")), "..", "data"))
    print(f"Local — BASE: {BASE}")

In [ ]:
!pip install kaleido==0.2.1

#2. Project Introduction

##2.1 Project Goals

This project investigates luxury private aviation(LPA) traffic patterns across Europe using flight-level data sourced from Eurocontrol's Open Performance Data Initiative (OPDI), covering the period January 2022 to December 2025.

The project applies a custom identification methodology to isolate LPA flights from the broader air traffic dataset using aircraft model information and ICAO type codes to construct a curated fleet list representative of the LPA segment. This filtering process addresses a gap in publicly available aviation analytics, where LPA traffic is rarely disaggregated from commercial and general aviation data at scale.

Primary goals of this project are:

- Conduct an extensive exploratory data analysis(EDA) of the identified LPA sample to understand the behavioral patterns of people with access to LPA, which is close to representative of rich and super rich. Understanding patterns of this segment is useful for luxury retail, charter flight, and investment companies for various business decisions.

- Develop an LPA non-domestic arrival forecasting model for a chosen region(France) over a 21-day horizon. This horizon was selected to align with the operational planning cycle of private charter operators and luxury ground service providers, for whom a three-week lead time represents the minimum actionable window for aircraft positioning, staffing decisions, and inventory procurement. Alongside modelling, the goal is also to analyze the effect of major local/nearby events on driving the arrival traffic.

##2.2 About Source And Data

Flights data with selected date range of all months from 2022 to 2025 was downloaded from https://www.eurocontrol.int/performance/data/download/OPDI/v002/flight_list/flight_list_ on 13.03.2026.

Eurocontrol operates Network Manager Operations Centre (NMOC) in Brussels. All flight plans going to and over Europe are submitted by pilots or operators to NMOC then receive flight validations or corrections based on its legal and operational validity alongside balancing aircraft demand with available airspace capacity. The Open Performance Data Initiative (OPDI) was launched in 2022 by the Performance Review Commission (PRC) of Eurocontrol with the objective of establishing a harmonized open data environment to support transparency, reproducibility, and accountability in European Air Traffic Management (ATM) performance. OPDI transforms raw ADS-B state vectors derived from the OpenSky Network into enriched flight datasets with geospatial indexing, event detection, and data quality monitoring.

As Eurcontrol's main mission is traffic control, it has some unnecessary columns for taxi and other purposes, hence selected columns to keep were:

*   `id`: unique identifier of the single flight
*   `icao24`: unique identifier of the aircraft
*   `dof`: day of flight
*   `adep`: aerodrome of departure
*   `ades`: aerodrome of destination
*   `model`: aircraft model name specifics
*   `typecode`: ICAO assigned aircraft designators that describe the plane's physical model.
*   `first_seen`: departure time, usually at the off-block time
*   `last_seen`: destination time, before in-block time


#3. LPA Data Preparation

##3.1 Eurocontrol Data Load And Check

###3.1.1. Monthly Data Load And Merge

In [ ]:
FLIGHTS_DIR = os.path.join(BASE, "Flights")
os.makedirs(FLIGHTS_DIR, exist_ok=True)

COLS     = ["id", "icao24", "dof", "adep", "ades", "typecode", "model", "first_seen", "last_seen"]
URL_BASE = "https://www.eurocontrol.int/performance/data/download/OPDI/v002/flight_list/flight_list_"

current_dt = datetime(2022, 1, 1)
end_dt     = datetime(2025, 12, 1)

while current_dt <= end_dt:
    month_str   = current_dt.strftime("%Y%m")
    target_file = os.path.join(FLIGHTS_DIR, f"flight_list_{month_str}.parquet")
    if not os.path.exists(target_file):
        try:
            resp = requests.get(f"{URL_BASE}{month_str}.parquet")
            resp.raise_for_status()
            df = pd.read_parquet(pd.io.common.BytesIO(resp.content), columns=COLS)
            df.to_parquet(target_file, index=False)
            print(f"Saved: {month_str}")
        except Exception as e:
            print(f"Error {month_str}: {e}")
    current_dt += relativedelta(months=1)

Before we do a merge to have all flights in one file, I want to check that all are schema-consistent.

In [ ]:
files        = sorted([f for f in os.listdir(FLIGHTS_DIR) if f.endswith(".parquet")])
meta         = pq.read_metadata(os.path.join(FLIGHTS_DIR, files[0])).schema
current_spec = {meta.names[i]: str(meta.column(i).logical_type) for i in range(len(meta.names))}

for f in files[1:]:
    meta     = pq.read_metadata(os.path.join(FLIGHTS_DIR, f)).schema
    new_spec = {meta.names[i]: str(meta.column(i).logical_type) for i in range(len(meta.names))}
    if new_spec != current_spec:
        for col, new_type in new_spec.items():
            if new_type != current_spec.get(col):
                print(f"{f}: {col} {current_spec.get(col)} -> {new_type}")
        current_spec = new_spec

We can see that from September 2025 there were some data type changes and December 2025 id column has changed too. We need to take these into consideration when creating a combined masterfile of all eurocontrol flights between 2022 and 2025.

In [ ]:
OUTPUT_FILE     = os.path.join(BASE, "all_eurocontrol_flights_2022_2025.parquet")
DATE_COLS       = ["dof", "first_seen", "last_seen"]
TRANSFORM_MONTHS = ["202509", "202510", "202511", "202512"]

files  = sorted([f for f in os.listdir(FLIGHTS_DIR) if f.endswith(".parquet")])
writer = None

for f in files:
    path       = os.path.join(FLIGHTS_DIR, f)
    file_month = f.split('_')[-1].split('.')[0]
    df         = pd.read_parquet(path, columns=COLS)

    if file_month in TRANSFORM_MONTHS:
        for col in DATE_COLS:
            df[col] = pd.to_datetime(df[col], errors='coerce')

    for col in DATE_COLS:
        df[col] = df[col].astype('datetime64[us]')

    df['id'] = df['id'].astype('uint64')
    table    = pa.Table.from_pandas(df, preserve_index=False)

    if writer is None:
        writer = pq.ParquetWriter(OUTPUT_FILE, table.schema, compression='snappy')
    writer.write_table(table)
    print(f"Merged: {f}")

if writer:
    writer.close()

###3.1.2. Data Check

Let's check how our masterfile looks like for now.

In [ ]:
FULL_OPDI_FILE = os.path.join(BASE, "all_eurocontrol_flights_2022_2025.parquet")
pf = pq.ParquetFile(FULL_OPDI_FILE)
head_data = next(pf.iter_batches(batch_size=10)).to_pandas()

print(head_data)

I already see from the data head that some fligt records don't have both departure and destination aerodromes. These records won't be useful in our research as I will not be able to make relevant meaningfull analyses. They are most likely just flights that were passing throuugh Eurocontrol area but didn't depart or land there. I will drop them. Cases where one is uknown, the other is known are still useful for the analysis, so they remain.

In [ ]:
TEMP_FILE = os.path.join(BASE, "temp_filtered.parquet")
writer = None

for batch in pf.iter_batches(batch_size=1_000_000):
    df = batch.to_pandas()
    df = df.dropna(subset=['adep', 'ades'], how='all')

    if not df.empty:
        table = pa.Table.from_pandas(df, preserve_index=False)
        if writer is None:
            writer = pq.ParquetWriter(TEMP_FILE, table.schema, compression='snappy')
        writer.write_table(table)

if writer:
    writer.close()
    os.replace(TEMP_FILE, FULL_OPDI_FILE)

Now let's check the data types, nulls, and unique counts for each column.

In [ ]:
meta = pf.metadata
schema = pf.schema

null_counts = {name: 0 for name in schema.names}
for i in range(pf.num_row_groups):
    rg = meta.row_group(i)
    for j in range(rg.num_columns):
        null_counts[schema.names[j]] += rg.column(j).statistics.null_count

unique_trackers = {name: set() for name in schema.names}
for batch in pf.iter_batches(batch_size=1_000_000):
    df_batch = batch.to_pandas()
    for col in schema.names:
        unique_trackers[col].update(df_batch[col].dropna().unique())

print(f"Rows: {meta.num_rows:,}")
print(f"{'Column':<12} | {'Nulls':<12} | {'Unique':<12} | {'Type'}")
print("-" * 65)

for i, name in enumerate(schema.names):
    l_type = str(schema.column(i).logical_type)
    u_count = len(unique_trackers[name])
    print(f"{name:<12} | {null_counts[name]:<12,} | {u_count:<12,} | {l_type}")

Everything looks fine, id is exactly as many as total rows, very important columns (icao24,dof,first_seen,last_seen) all don't have null values, whle textual columns seem to have quite some null values but it's okay. The critical insight I notice is that dof are 1442, while there are 1461 days in our chosen data period. This signals data gaps. I want to check all possible source data gap by first looking at daily total aviation traffic.

In [ ]:
FULL_OPDI_FILE = os.path.join(BASE, "all_eurocontrol_flights_2022_2025.parquet")
df_all = pd.read_parquet(FULL_OPDI_FILE, columns=['dof'])

df_all['dof'] = pd.to_datetime(df_all['dof'])
daily_counts = df_all.groupby('dof').size()

full_range = pd.date_range(start='2022-01-01', end='2025-12-31')
ts = daily_counts.reindex(full_range, fill_value=0).reset_index()
ts.columns = ['date', 'count']

In [ ]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=ts['date'],
    y=ts['count'],
    mode='lines',
    name='Daily Flight Count',
    line=dict(color='#808080', width=1.5),
    fill='tozeroy',
    fillcolor='rgba(128, 128, 128, 0.1)'
))

fig.update_layout(
    title=dict(
        text="<b>Total Daily Aviation Traffic (2022-2025, Eurocontrol Area)</b><br><sup>Source: Author's own construction based on Eurocontrol OPDI data</sup>",
        x=0.5,
        font=dict(color="black", size=16)
    ),
    xaxis=dict(
        title="<b>TIMELINE</b>",
        title_font=dict(size=11),
        showgrid=False,
        linecolor='black',
        linewidth=1.2,
        tickfont=dict(size=10, color='black')
    ),
    yaxis=dict(
        title="<b>NUMBER OF FLIGHTS</b>",
        title_font=dict(size=11),
        showgrid=True,
        gridcolor='rgba(0,0,0,0.05)',
        linecolor='black',
        linewidth=1.2,
        tickfont=dict(size=10, color='black')
    ),
    paper_bgcolor="white",
    plot_bgcolor="white",
    height=600,
    width=1200,
    margin=dict(t=100, b=50, l=50, r=50),
    hovermode="x unified",
    showlegend=False
)

fig.show()

There are quite several suspicuous days which have zero or much lower than usual minimum volume. I want to flag them for later reference, using condition lower than 10k flights in a day which are the most suspicuous unrealistic/unusual records.

In [ ]:
possible_data_gaps_df = ts[ts['count'] < 10000].sort_values(by='date')
possible_data_gaps = possible_data_gaps_df['date'].dt.strftime('%Y-%m-%d').tolist()

print(f"Detected {len(possible_data_gaps)} possible data gaps:")
print(possible_data_gaps)

And lastly, I want to ensure that no empty spaces are being treated as non null values

In [ ]:
STR_COLS = ["icao24", "adep", "ades", "typecode", "model"]

found = False

for batch in pf.iter_batches(columns=STR_COLS, batch_size=1_000_000):
    df = batch.to_pandas()
    for col in STR_COLS:
        artifacts = df[col].dropna()
        matches = artifacts.str.fullmatch(r'\s*')

        if matches.any():
            print(f"Artifact found in {col}: '{artifacts[matches].iloc[0]}'")
            found = True
            break
    if found: break

if not found:
    print("No whitespace artifacts detected. All missing data are true logical nulls.")

##3.2 LPA Identification

In order to find luxury private aviation flights, I need to create file of all unique aircrafts present in our flights data alongside with typecode and model values.

In [ ]:
AIRCRAFT_FILE = os.path.join(BASE, "all_eurocontrol_flights_2022_2025_aircraft_data.parquet")
FULL_OPDI_FILE = os.path.join(BASE, "all_eurocontrol_flights_2022_2025.parquet")

pf = pq.ParquetFile(FULL_OPDI_FILE)
COLS = ["icao24", "typecode", "model"]

frames = []
for batch in pf.iter_batches(columns=COLS, batch_size=2_000_000):
    df_batch = batch.to_pandas().dropna(subset=["icao24"])
    frames.append(df_batch.drop_duplicates())

aircraft_df = pd.concat(frames).drop_duplicates()
aircraft_df = aircraft_df.sort_values(by=COLS, ascending=False).drop_duplicates(subset=["icao24"])

table = pa.Table.from_pandas(aircraft_df, preserve_index=False)
pq.write_table(table, AIRCRAFT_FILE, compression='snappy')

In [ ]:
AIRCRAFT_FILE = os.path.join(BASE, "all_eurocontrol_flights_2022_2025_aircraft_data.parquet")
pf = pq.ParquetFile(AIRCRAFT_FILE)
df = pf.read(columns=["typecode", "model"]).to_pandas()

cols = ["typecode", "model"]
print(f"Total Aircraft: {len(df):,}\n")
print(f"{'Column':<12} | {'Nulls':<12} | {'Unique':<12}")
print("-" * 40)

for name in cols:
    null_count = df[name].isna().sum()
    u_count = df[name].nunique()
    print(f"{name:<12} | {null_count:<12,} | {u_count:<12,}")

We have 65,996 unique aircrafts in our total flights data. I will try to add missing typecode and model data from external source - https://opensky-network.org/datasets/metadata/#metadata/ (aircraftdatabase-complete-2025-08.csv) - accessed and downloaded on 13.03.2026, so I can label as much luxury private aviation as it is possible under given time and resource constraints.

In [ ]:
EXT_FILE = os.path.join(BASE, "ext_aircraft-database-complete-2025-08.csv")

df_air = pq.read_table(AIRCRAFT_FILE).to_pandas()

df_ext = pd.read_csv(EXT_FILE, usecols=['icao24', 'typecode', 'model', 'owner'])
df_ext = df_ext.rename(columns={
    'typecode': 'ext_typecode',
    'model': 'ext_model',
    'owner': 'ext_owner',
})

df_ext = df_ext.drop_duplicates(subset=['icao24'])

df_final = pd.merge(df_air, df_ext, on='icao24', how='left')

table = pa.Table.from_pandas(df_final, preserve_index=False)
pq.write_table(table, AIRCRAFT_FILE, compression='snappy')

In [ ]:
pf = pq.ParquetFile(AIRCRAFT_FILE)
df = pf.read().to_pandas()

print(f"{'Column':<15} | {'Null Counts'}")
print("-" * 30)

for col in df.columns:
    print(f"{col:<15} | {df[col].isna().sum():,}")

It seems external source had more aircraft data. So I will use it to add where original columns are missing.

In [ ]:
df['typecode'] = df['typecode'].fillna(df['ext_typecode'])
df['model'] = df['model'].fillna(df['ext_model'])

df = df.drop(columns=['ext_typecode', 'ext_model'])

table = pa.Table.from_pandas(df, preserve_index=False)
pq.write_table(table, AIRCRAFT_FILE, compression='snappy')

print(f"{'Column':<15} | {'Null Counts'}")
print("-" * 30)
for col in df.columns:
    print(f"{col:<15} | {df[col].isna().sum():,}")

From 12k+ missing typecodes it fell down to 8k+, and for model it decreased by  4k missing values, so it was a good addition from external source. Now I can use the custom built typecodes list that represent majority of LPA, alongside some keywords in model column for cases when the typecode can be both commerical airliner plane and luxury private jet (Airbus & Boeing Corporate Jets)

In [ ]:
lpa_typecode_map = {
    'Gulfstream Classic':         ['G100', 'G150', 'G159', 'G280', 'ASTR', 'GALX', 'GLF2', 'GLF3', 'GLF4', 'GLF5', 'GLF6'],
    'Gulfstream Ultra-Long Range':['GA3C', 'GA4C', 'GA5C', 'GA6C', 'GA7C', 'GA8C'],
    'Bombardier Learjet':         ['LJ23', 'LJ24', 'LJ25', 'LJ28', 'LJ31', 'LJ35', 'LJ40', 'LJ45', 'LJ55', 'LJ60', 'LJ70', 'LJ75'],
    'Bombardier Challenger':      ['CL30', 'CL35', 'CL60', 'CL61', 'CL64', 'CL65'],
    'Bombardier Global':          ['GLEX', 'GL5T', 'GL6T', 'GL7T', 'GL8T'],
    'Dassault Falcon':            ['FA6X', 'FA7X', 'FA8X', 'F10X', 'FA10', 'FA20', 'FA50', 'F2TH', 'F900'],
    'Cessna Citation':            ['C500', 'C501', 'C510', 'C525', 'C550', 'C551', 'C560', 'C650', 'C680', 'C700', 'C750', 'C25A', 'C25M', 'C25B', 'C25C', 'C56X', 'C68A'],
    'Embraer Phenom/Praetor':     ['E50P', 'E55P', 'E545', 'E550', 'P500', 'P600'],
}

typecode_to_group = {tc: group for group, tcs in lpa_typecode_map.items() for tc in tcs}

def assign_lpa_group(row):
    tc = str(row['typecode']).strip().upper() if pd.notna(row['typecode']) else ''
    if tc in typecode_to_group:
        return typecode_to_group[tc]
    model = str(row['model']).strip() if pd.notna(row['model']) else ''
    model_upper = model.upper()
    if any(kw in model_upper for kw in ['AIRBUS CORPORATE JET', 'ACJ']):
        return 'Airbus Corporate Jet (ACJ)'
    if any(kw in model_upper for kw in ['BOEING CORPORATE JET', 'BCJ', 'BBJ']):
        return 'Boeing Corporate Jet (BCJ)'
    return 'non_lpa'

df = pq.read_table(AIRCRAFT_FILE).to_pandas()
df['lpa_group'] = df.apply(assign_lpa_group, axis=1)

print(df['lpa_group'].value_counts())

table = pa.Table.from_pandas(df, preserve_index=False)
pq.write_table(table, AIRCRAFT_FILE, compression='snappy')

Now that I have marked luxury private aircrafts, I can compile a file of only luxury private aviation from all_eurocontrol_flights_2022_2025.pq file by only including those records where icao24 was identified as private luxury aircraft in the aircraft file.

In [ ]:
FULL_OPDI_FILE = os.path.join(BASE, "all_eurocontrol_flights_2022_2025.parquet")
AIRCRAFT_FILE = os.path.join(BASE, "all_eurocontrol_flights_2022_2025_aircraft_data.parquet")
LPA_FILE       = os.path.join(BASE, "lpa_eurocontrol_flights_2022_2025.parquet")

df_aircraft = pq.read_table(AIRCRAFT_FILE).to_pandas()
df_lpa_aircraft = df_aircraft[df_aircraft['lpa_group'] != 'non_lpa'][['icao24', 'typecode', 'model', 'ext_owner', 'lpa_group']]

df_flights = pq.read_table(FULL_OPDI_FILE).to_pandas()
df_flights = df_flights.drop(columns=['typecode', 'model'], errors='ignore')

df_lpa = df_flights.merge(df_lpa_aircraft, on='icao24', how='inner')

print(f"Total flights:     {len(df_flights):,}")
print(f"LPA flights:       {len(df_lpa):,}")
print(f"LPA aircraft:      {df_lpa['icao24'].nunique():,}")

table = pa.Table.from_pandas(df_lpa, preserve_index=False)
pq.write_table(table, LPA_FILE, compression='snappy')


In [ ]:
pf = pq.ParquetFile(LPA_FILE)
df = pf.read().to_pandas()
print(f"Rows: {df.shape[0]:,}, Columns: {df.shape[1]}")
print(df.columns.tolist())

We have identified over 2m LPA traffic data. As noticed earlier however - model , typecode, and ext_owner values have null cases - I want to label them Unknown for later reference.

In [ ]:
for col in ['typecode', 'model', 'ext_owner']:
    df[col] = df[col].fillna('Unknown')

df.to_parquet(LPA_FILE, index=False)
print(f"Done. Rows: {len(df):,}")

I also want to check that model matching didn't catch any non Airbus or Boeing models

In [ ]:
df_aircraft = pq.read_table(AIRCRAFT_FILE).to_pandas()
acj_bcj = df_aircraft[df_aircraft['lpa_group'].isin(['Airbus Corporate Jet (ACJ)', 'Boeing Corporate Jet (BCJ)'])]
print(acj_bcj.groupby('lpa_group')['model'].value_counts().to_string())

All looks correct, it was successful identification.

##3.3 LPA Data Exension With Aerodrome Specifics

Now that we have LPA flight records, I would like to add information on destination and departure aerodromes so that I can analyze flights geospatially. I accessed and downloaded airports data from https://ourairports.com/data/ on 13.03.2026, creating a single ext_airports.csv file by combining files airports.csv, countries.csv, regions.csv and removing unnecessary columns. I will check first if all ades and adep values can find the matches, and see the codes that didn't find matches.

In [ ]:
LPA_FILE = os.path.join(BASE, "lpa_eurocontrol_flights_2022_2025.parquet")
EXT_AIRPORTS = os.path.join(BASE, "ext_airports.csv")

df_ext = pd.read_csv(EXT_AIRPORTS, usecols=['airport_icao'])
reference_set = set(df_ext['airport_icao'].dropna().unique())

pf = pq.ParquetFile(LPA_FILE)
flight_codes = set()

for batch in pf.iter_batches(columns=['adep', 'ades'], batch_size=1_000_000):
    df_batch = batch.to_pandas()
    flight_codes.update(df_batch['adep'].dropna().unique())
    flight_codes.update(df_batch['ades'].dropna().unique())

missing_codes = sorted(list(flight_codes - reference_set))

print(f"Total unique aerodromes: {len(flight_codes):,}")
print(f"Total matched: {len(flight_codes & reference_set):,}")
print(f"Total missing: {len(missing_codes):,}")
print("-" * 40)
print(missing_codes)

There are missing data for 16 aerodromes, so I manually constructed the list with their data using online search to make sure all aerodromes find match. For cases where ades/adep is null I will label all textual information as "Unknown/Outside Wider Europe". From the external data I will add airport name (name), municipality, country name, latitude, and longitude. For matches I will use adep_ and ades_ prefixes so it adds airports values for both departure and destination to clearly define to which part the of the flight the aerodrome specifics relate to.

In [ ]:
LPA_FILE = os.path.join(BASE, "lpa_eurocontrol_flights_2022_2025.parquet")
EXT_AIRPORTS = os.path.join(BASE, "ext_airports.csv")
FULL_LPA_FILE = os.path.join(BASE, "full_lpa_eurocontrol_flights_2022_2025.parquet")

TEXT_COLS = ['name', 'municipality', 'country_name']
NUM_COLS = ['latitude_deg', 'longitude_deg']
ALL_TARGETS = TEXT_COLS + NUM_COLS

manual_data = """airport_icao,name,municipality,country_name,latitude_deg,longitude_deg
CY-0002,Lefkoniko Airport / Geçitkale Air Base,Geçitkale (Lefkoniko),Cyprus,35.23583,33.72417
EGXP,RAF Scampton,Scampton,United Kingdom,53.30780,-0.55083
EVDA,Daugavpils Airport,Daugavpils,Latvia,55.94472,26.66500
LFTO,Rize–Artvin Airport,Rize,Turkey,41.16917,40.82889
LROV,Brașov-Ghimbav International Airport,Brașov,Romania,45.70558,25.52287
LY88,Ponikve Airport,Užice,Serbia,43.89890,19.69770
OKBK,Kuwait International Airport,Kuwait City,Kuwait,29.22660,47.96890
RU-0035,Grozny Airport,Grozny,Russia,43.38820,45.69990
TR-0044,Çukurova Regional Airport,Tarsus,Turkey,36.89083,35.07028
ULAK,Severomorsk-1 Naval Air Base,Severomorsk,Russia,69.03170,33.41833
UUBL,Semyazino Airport,Vladimir,Russia,56.12700,40.39800
UUBM,Myachkovo Airport,Moscow,Russia,55.56076,37.98368
UUWE,Yermolino Air Base,Yermolino,Russia,55.22833,36.60833
XLLL,Soltsy-2 Air Base,Soltsy,Russia,58.13911,30.32880
XLMV,Severomorsk-3 Naval Air Base,Severomorsk,Russia,68.86686,33.71821
XUBS,Smolensk North Airport,Smolensk,Russia,54.82500,32.02500"""

df_ext = pd.read_csv(EXT_AIRPORTS)[['airport_icao'] + ALL_TARGETS]
df_manual = pd.read_csv(StringIO(manual_data))
df_combined = pd.concat([df_ext, df_manual]).drop_duplicates(subset=['airport_icao'])

def get_lookup(prefix):
    lookup = df_combined[['airport_icao'] + ALL_TARGETS].copy()
    lookup.columns = [prefix] + [f"{prefix}_{col}" for col in ALL_TARGETS]
    return lookup

df_adep_lookup, df_ades_lookup = get_lookup('adep'), get_lookup('ades')

pf = pq.ParquetFile(LPA_FILE)
writer = None

for batch in pf.iter_batches(batch_size=1_000_000):
    df_batch = batch.to_pandas()
    df_batch['adep'] = df_batch['adep'].fillna("Unknown/Outside Wider Europe")
    df_batch['ades'] = df_batch['ades'].fillna("Unknown/Outside Wider Europe")
    df_batch = pd.merge(df_batch, df_adep_lookup, on='adep', how='left')
    df_batch = pd.merge(df_batch, df_ades_lookup, on='ades', how='left')
    for prefix in ['adep', 'ades']:
        for col in TEXT_COLS:
            df_batch[f"{prefix}_{col}"] = df_batch[f"{prefix}_{col}"].fillna("Unknown/Outside Wider Europe")
    table = pa.Table.from_pandas(df_batch, preserve_index=False)
    if writer is None:
        writer = pq.ParquetWriter(FULL_LPA_FILE, table.schema, compression='snappy')
    writer.write_table(table)

if writer:
    writer.close()

In [ ]:
FULL_LPA_FILE = os.path.join(BASE, "full_lpa_eurocontrol_flights_2022_2025.parquet")
df = pd.read_parquet(FULL_LPA_FILE)

print(f"Total Rows: {len(df):,}")
print(f"\n{'Column':<20} | {'Type':<20} | {'Null Counts'}")
print("-" * 45)
for col in df.columns:
    print(f"{col:<20} | {str(df[col].dtype):<20} | {df[col].isna().sum():,}")

Everything looks good, I don't want to replace missing coordinates for unknown ades and adep with unknown label as it will change the entire column data type, so I am content with the final LPA data. We can also conclude that we have 1,824,045 LPA flights in total and as we deleted previously records with both unknown adep and ades, we have:
- 533,266 flights with unknown departure aerodrome (but known destination)
- 334,433 flights with unknown destination aerodrome(but known departure)
- and remaining flights have complete aerodrome data.

In [ ]:
for col in ['adep_municipality', 'ades_municipality', 'adep_country_name', 'ades_country_name']:
    print(f"{col}: {df[col].nunique():,} unique")

all_municipalities = pd.concat([df['adep_municipality'], df['ades_municipality']]).nunique()
all_countries      = pd.concat([df['adep_country_name'], df['ades_country_name']]).nunique()

print(f"\nTotal unique municipalities (adep + ades combined): {all_municipalities:,}")
print(f"Total unique countries      (adep + ades combined): {all_countries:,}")

#4. LPA EDA

In [ ]:
FULL_LPA_FILE = os.path.join(BASE, "full_lpa_eurocontrol_flights_2022_2025.parquet")
df = pd.read_parquet(FULL_LPA_FILE)

##4.1 LPA Traffic Volume and Seasonality

First things first, I want to check the daily traffic of luxury private aviation. I will also include to show earlier identified possible source data gap days as dot lines.

In [ ]:
df['dof'] = pd.to_datetime(df['dof'])
daily_counts = df.groupby('dof').size()

full_range = pd.date_range(start='2022-01-01', end='2025-12-31')
ts_lpa = daily_counts.reindex(full_range, fill_value=0).reset_index()
ts_lpa.columns = ['date', 'count']

In [ ]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=ts['date'],
    y=ts['count'],
    mode='lines',
    name='Total Aviation Traffic',
    line=dict(color='#808080', width=1.5),
    fill='tozeroy',
    fillcolor='rgba(128, 128, 128, 0.1)'
))

fig.add_trace(go.Scatter(
    x=ts_lpa['date'],
    y=ts_lpa['count'],
    mode='lines',
    name='LPA* Traffic',
    line=dict(color='#F9E076', width=1.5),
    fill='tozeroy',
    fillcolor='rgba(245, 225, 164, 0.3)'
))

for date in possible_data_gaps:
    fig.add_vline(
        x=date,
        line_width=1,
        line_dash="dot",
        line_color="#B9B9B9",
        opacity=0.6
    )

fig.add_trace(go.Scatter(
    x=[None], y=[None],
    mode='lines',
    name='Source Data Gaps',
    line=dict(color='#B9B9B9', width=1, dash='dot'),
    showlegend=True
))

fig.update_layout(
    title=dict(
        text="<b>Total Daily Aviation vs LPA* Traffic (2022–2025, Eurocontrol Area)</b><br>"
             "<sup>Source: Author's own construction based on Eurocontrol OPDI data</sup>",
        x=0.5,
        font=dict(color="black", size=16)
    ),

    xaxis=dict(
        title="<b>TIMELINE</b>",
        title_font=dict(size=11),
        showgrid=False,
        linecolor='black',
        linewidth=1.2,
        tickfont=dict(size=10, color='black')
    ),
    yaxis=dict(
        title="<b>NUMBER OF FLIGHTS</b>",
        title_font=dict(size=11),
        showgrid=True,
        gridcolor='rgba(0,0,0,0.05)',
        linecolor='black',
        linewidth=1.2,
        tickfont=dict(size=10, color='black')
    ),
    showlegend=True,
    legend=dict(
        orientation="h",
        yanchor="top",
        y=-0.12,
        xanchor="center",
        x=0.5,
        font=dict(size=10)
    ),
    paper_bgcolor="white",
    plot_bgcolor="white",
    height=500,
    width=1000,
    margin=dict(t=100, b=100, l=50, r=50),
    hovermode="x unified",
    annotations=[dict(
        text="LPA* – Luxury Private Aviation",
        xref="paper", yref="paper",
        x=1, y=-0.2,
        xanchor="right", yanchor="bottom",
        showarrow=False,
        font=dict(size=9, color="grey"),
    )]
)

fig.show()
fig.write_image("Figure 1. Total Daily Aviation vs LPA Traffic (2022-2025, Eurocontrol Area).svg")

In [ ]:
merged = ts.merge(ts_lpa, on='date', suffixes=('_total', '_lpa'))
merged['pct'] = merged['count_lpa'] / merged['count_total'] * 100
print(f"Average daily LPA share of total aviation: {merged['pct'].mean():.2f}%")

In [ ]:
print(f"Max total aviation day: {ts.loc[ts['count'].idxmax(), 'date'].date()}  —  {ts['count'].max():,} flights")
print(f"Max LPA aviation day:   {ts_lpa.loc[ts_lpa['count'].idxmax(), 'date'].date()}  —  {ts_lpa['count'].max():,} flights")

In [ ]:
df = pd.read_parquet(FULL_LPA_FILE, columns=['icao24', 'adep_country_name', 'ades_country_name'])
is_unk = (df['adep_country_name'] == "Unknown/Outside Wider Europe") | (df['ades_country_name'] == "Unknown/Outside Wider Europe")
is_nat = (~is_unk) & (df['adep_country_name'] == df['ades_country_name'])
is_int = (~is_unk) & (df['adep_country_name'] != df['ades_country_name'])

flight_dist_data = {
    'Labels': ['National', 'International Within Wider Europe', 'From or To Unknown/International Outside Wider Europe'],
    'Values': [is_nat.sum(), is_int.sum(), is_unk.sum()]
}
fig = go.Figure()

fig.add_trace(go.Pie(
    labels=flight_dist_data['Labels'],
    values=flight_dist_data['Values'],
    hole=0.4,
    marker=dict(
        colors=['#f8da5e', '#F9E076', '#B9B9B9'],
        line=dict(color='white', width=2)
    ),
    textinfo='percent+label',
    textposition='outside',
    insidetextorientation='radial',
    textfont=dict(size=11, color='black')
))

fig.update_layout(
    title=dict(
        text="<b>LPA* Traffic Composition: International vs National (2022-2025, Eurocontrol Area) </b><br><sup>Source: Author\'s own analysis based on Eurocontrol OPDI data </sup>",
        x=0.5,
        font=dict(color="black", size=16)
    ),
    showlegend=False,
    paper_bgcolor="white",
    plot_bgcolor="white",
    height=550,
    width=1000,
    margin=dict(t=100, b=50, l=50, r=50),
    annotations=[dict(
        text="LPA* – Luxury Private Aviation",
        xref="paper", yref="paper",
        x=1.6, y=-0.9,
        xanchor="right", yanchor="bottom",
        showarrow=False,
        font=dict(size=9, color="grey"),
    )]
)

fig.show()

In [ ]:
ts_lpa['dayofweek'] = ts_lpa['date'].dt.day_name()
ts_lpa['month']     = ts_lpa['date'].dt.strftime('%b')
ts_lpa['quarter']   = 'Q' + ts_lpa['date'].dt.quarter.astype(str)
ts_lpa['year']      = ts_lpa['date'].dt.year.astype(str)

day_order     = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
month_order   = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
quarter_order = ['Q1','Q2','Q3','Q4']

fig = make_subplots(rows=2, cols=2,
    subplot_titles=[
        'Weekly Seasonality',
        'Monthly Seasonality',
        'Quarterly Seasonality',
        'Yearly Distribution'
    ],
    vertical_spacing=0.15,
    horizontal_spacing=0.1
)

for day in day_order:
    vals = ts_lpa[ts_lpa['dayofweek'] == day]['count']
    fig.add_trace(go.Box(
        y=vals, name=day, showlegend=False,
        marker_color='#F9E076'
    ), row=1, col=1)

for month in month_order:
    vals = ts_lpa[ts_lpa['month'] == month]['count']
    fig.add_trace(go.Box(
        y=vals, name=month, showlegend=False,
        marker_color='#bab9b5'
    ), row=1, col=2)

for q in quarter_order:
    vals = ts_lpa[ts_lpa['quarter'] == q]['count']
    fig.add_trace(go.Box(
        y=vals, name=q, showlegend=False,
        marker_color='#B9B9B9'
    ), row=2, col=1)

for year in sorted(ts_lpa['year'].unique()):
    vals = ts_lpa[ts_lpa['year'] == year]['count']
    fig.add_trace(go.Box(
        y=vals, name=year, showlegend=False,
        marker_color='#b8ac7b'
    ), row=2, col=2)

fig.update_layout(
    title=dict(
        text='<b>LPA Traffic Seasonality — Weekly, Monthly, Quarterly & Yearly</b>',
        x=0.5, font=dict(size=16)
    ),
    template='plotly_white',
    height=800,
    margin=dict(t=100, l=50, r=50, b=50)
)
fig.show()

We can see observe:

- weekly seasonality - specifically weekdays having more traffic than weekends.
- monthly seasonality - May, June, July, September being highest traffic months.
- quarterly seasonality - 2&3 quarters having higher traffic than the rest
- no one upwards or downward trend over time, as 2022 had higest distribution and average traffic, then went down, and from 2023 started growing slightly year by year.

We can also check by week of the year alongside the day of the week instead of month or quarter, to see more granular seasonality patterns.

In [ ]:
ts_lpa['week'] = ts_lpa['date'].dt.isocalendar().week.astype(int)
ts_lpa['dow']  = ts_lpa['date'].dt.dayofweek

heatmap_data  = ts_lpa.groupby(['week', 'dow'])['count'].sum().reset_index()
heatmap_pivot = heatmap_data.pivot(index='dow', columns='week', values='count').fillna(0)

day_labels = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']

fig = go.Figure(go.Heatmap(
    z=heatmap_pivot.values,
    x=heatmap_pivot.columns.tolist(),
    y=day_labels,
    colorscale=[[0, '#fefcf0'], [0.5, '#f6cf2d'], [1, '#f27907']],
    hovertemplate='Week: %{x}<br>Day: %{y}<br>Total Flights: %{z}<extra></extra>',
    colorbar=dict(title='Total Flights')
))

for w in [1, 14, 27, 40]:
    fig.add_vline(x=w - 0.5, line_dash='dash', line_color='gray',
                  line_width=1, opacity=0.5)

for q, w in {'Q1': 7, 'Q2': 20, 'Q3': 33, 'Q4': 46}.items():
    fig.add_annotation(
        x=w,
        text=f'<b>{q}</b>',
        showarrow=False,
        font=dict(size=11, color='gray'),
        yref='paper', y=1
    )

fig.update_layout(
    font=dict(color="black"),
    title=dict(
        text='<b>LPA* Traffic Weekly Seasonality (2022–2025, Eurocontrol Area)</b><br>'
             '<sup>Source: Author\'s own construction based on Eurocontrol OPDI data</sup>',
        x=0.5, font=dict(size=16, color="black")
    ),
    xaxis=dict(title='Week of Year', tickmode='linear', dtick=2, title_font=dict(size=11),tickfont=dict(size=10, color='black')
),
    yaxis=dict(title='Day of Week',title_font=dict(size=11),tickfont=dict(size=10, color='black')),
    paper_bgcolor='white',
    plot_bgcolor='white',
    height=500,
    width=1000,
    margin=dict(t=100, l=80, r=50, b=80)
)

fig.update_layout(
    annotations=[dict(
        text="LPA* – Luxury Private Aviation",
        xref="paper", yref="paper",
        x=1, y=-0.2,
        xanchor="right", yanchor="bottom",
        showarrow=False,
        font=dict(size=9, color="grey"),
    )]
)
fig.show()
fig.write_image("Figure 2. LPA Traffic Weekly Seasonality (2022-2025, Eurocontrol Area).svg")

Heatmap efficiently shows how much saturdays are lower than the rest, and also how much first and last weeks of the year differ from the center peak weeks 22-28 alongside secondary peak in late Augest to early October.

In [ ]:
df_hours = pd.read_parquet(FULL_LPA_FILE, columns=['first_seen'])
df_hours['first_seen'] = pd.to_datetime(df_hours['first_seen'])
df_hours['dep_hour'] = df_hours['first_seen'].dt.hour
df_hours['dow']      = df_hours['first_seen'].dt.dayofweek

dep_heat  = df_hours.groupby(['dow', 'dep_hour']).size().reset_index(name='count')
dep_pivot = dep_heat.pivot(index='dow', columns='dep_hour', values='count').fillna(0)

day_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']

fig = go.Figure(go.Heatmap(
    z=dep_pivot.values,
    x=[f'{h:02d}:00' for h in dep_pivot.columns],
    y=day_labels,
    colorscale=[[0, '#fefcf0'], [0.5, '#f6cf2d'], [1, '#f27907']],
    hovertemplate='Day: %{y}<br>Hour: %{x}<br>Flights: %{z}<extra></extra>',
    colorbar=dict(title='Total Flights')
))

fig.update_layout(
    font=dict(color='black'),
    title=dict(
        text='<b>LPA* Departure Hour by Day of Week (2022–2025, Eurocontrol Area)</b><br>'
             '<sup>Source: Author\'s own construction based on Eurocontrol OPDI data</sup>',
        x=0.5, font=dict(size=16, color='black')
    ),
    xaxis=dict(
        title='<b>HOUR OF DAY (UTC)</b>',
        title_font=dict(size=11),
        tickfont=dict(size=10, color='black'),
        linecolor='black', linewidth=1.2
    ),
    yaxis=dict(
        title='<b>DAY OF WEEK</b>',
        title_font=dict(size=11),
        tickfont=dict(size=10, color='black'),
        linecolor='black', linewidth=1.2
    ),
    paper_bgcolor='white',
    plot_bgcolor='white',
    height=400,
    width=1000,
    margin=dict(t=100, b=80, l=80, r=50),
    annotations=[dict(
        text="LPA* – Luxury Private Aviation",
        xref="paper", yref="paper",
        x=1, y=-0.18,
        xanchor="right", yanchor="bottom",
        showarrow=False,
        font=dict(size=9, color="grey"),
    )]
)

fig.show()
fig.write_image("Figure 3. LPA Departure Hour by Day of Week (2022-2025, Eurocontrol Area).svg")

##4.2 LPA Fleet Analysis

###4.2.1. LPA Fleet Composition And Trend

In [ ]:
df_flights_total = pd.read_parquet(FULL_LPA_FILE, columns=['icao24', 'dof'])
df_flights_total['dof'] = pd.to_datetime(df_flights_total['dof'])
df_flights_total['year'] = df_flights_total['dof'].dt.year

all_years = sorted(df_flights_total['year'].unique())
seen = set()
total_cumulative = []
new_entrants = []

for year in all_years:
    year_aircraft = set(df_flights_total[df_flights_total['year'] == year]['icao24'].unique())
    new = year_aircraft - seen
    seen.update(year_aircraft)
    total_cumulative.append({'year': year, 'cumulative': len(seen)})
    new_entrants.append({'year': year, 'new_entrants': len(new)})

df_cumulative   = pd.DataFrame(total_cumulative)
df_new_entrants = pd.DataFrame(new_entrants)
df_new_entrants = df_new_entrants[df_new_entrants['year'] > 2022]

fig = make_subplots(rows=1, cols=2,
    column_widths=[0.5, 0.5],
    specs=[[{"type": "scatter"}, {"type": "bar"}]])

fig.add_trace(go.Scatter(
    x=df_cumulative['year'],
    y=df_cumulative['cumulative'],
    mode='lines+markers+text',
    name='Cumulative Fleet',
    line=dict(color='#F8DA5E', width=2),
    marker=dict(size=8, color='#F8DA5E', line=dict(color='#666666', width=0.8)),
    text=df_cumulative['cumulative'].apply(lambda x: f'{x:,}'),
    textposition='top center',
    textfont=dict(size=10, color='black'),
    hovertemplate='Year: %{x}<br>Total Fleet: %{y:,}<extra></extra>'
), row=1, col=1)

fig.add_trace(go.Bar(
    x=df_new_entrants['year'],
    y=df_new_entrants['new_entrants'],
    name='New Entrants',
    marker_color='#f27907',
    marker_line=dict(color='#666666', width=0.8),
    text=df_new_entrants['new_entrants'].apply(lambda x: f'{x:,}'),
    textposition='outside',
    textfont=dict(size=10, color='black'),
    hovertemplate='Year: %{x}<br>New Aircraft: %{y:,}<extra></extra>'
), row=1, col=2)



fig.update_layout(
    font=dict(color='black'),
    title=dict(
        text='<b>LPA* Total Fleet Growth & New Entrants by Year (2022–2025, Eurocontrol Area)</b><br>'
             '<sup>Source: Author\'s own construction based on Eurocontrol OPDI data</sup>',
        x=0.5, y=0.89,
        font=dict(size=16, color='black')
    ),
    paper_bgcolor='white',
    plot_bgcolor='white',
    height=500,
    width=1100,
    margin=dict(t=130, b=80, l=70, r=50),
    showlegend=False,
    xaxis=dict(
        title='<b>YEAR</b>', title_font=dict(size=11),
        tickmode='linear', dtick=1,
        tickfont=dict(size=10, color='black'),
        linecolor='black', linewidth=1.2
    ),
    yaxis=dict(
        title='<b>CUMULATIVE UNIQUE AIRCRAFT</b>', title_font=dict(size=11),
        tickfont=dict(size=10, color='black'),
        linecolor='black', linewidth=1.2,
        showgrid=True, gridcolor='rgba(0,0,0,0.05)'
    ),
    xaxis2=dict(
        title='<b>YEAR</b>', title_font=dict(size=11),
        tickmode='linear', dtick=1,
        tickfont=dict(size=10, color='black'),
        linecolor='black', linewidth=1.2
    ),
    yaxis2=dict(
        title='<b>NEW AIRCRAFT ENTRANTS</b>', title_font=dict(size=11),
        tickfont=dict(size=10, color='black'),
        linecolor='black', linewidth=1.2,
        showgrid=True, gridcolor='rgba(0,0,0,0.05)'
    ),
    annotations=[
        dict(text='<b>Cumulative Fleet Size</b>', xref="paper", yref="paper",
             x=0.22, y=1.02, xanchor="center", yanchor="bottom",
             showarrow=False, font=dict(size=13, color="black")),
        dict(text='<b>New Aircraft Entrants per Year</b>', xref="paper", yref="paper",
             x=0.78, y=1.02, xanchor="center", yanchor="bottom",
             showarrow=False, font=dict(size=13, color="black")),
        dict(text="LPA* – Luxury Private Aviation", xref="paper", yref="paper",
             x=1, y=-0.12, xanchor="right", yanchor="bottom",
             showarrow=False, font=dict(size=9, color="grey")),
    ]
)

fig.show()
fig.write_image("Figure 4. LPA Total Fleet Growth and New Entrants (2022-2025, Eurocontrol Area).svg")

In [ ]:
lpa_group_colors = {
    'Bombardier Learjet':           '#D4D4D4',
    'Bombardier Challenger':        '#B9B9B9',
    'Bombardier Global':            '#9E9E9E',
    'Gulfstream Classic':           '#FBEDA0',
    'Gulfstream Ultra-Long Range':  '#F8DA5E',
    'Dassault Falcon':              '#f27907',
    'Airbus Corporate Jet (ACJ)':   '#F9E076',
    'Boeing Corporate Jet (BCJ)':   '#F5CE3A',
    'Embraer Phenom/Praetor':       '#635a52',
    'Cessna Citation':              '#e8a86f',
}

df_lpa_only = (pq.read_table(FULL_LPA_FILE, columns=['icao24', 'lpa_group'])
                 .to_pandas()
                 .drop_duplicates(subset='icao24'))

fleet_comp = df_lpa_only.groupby('lpa_group')['icao24'].nunique().reset_index()
fleet_comp.columns = ['lpa_group', 'aircraft_count']
fleet_comp = fleet_comp.sort_values('aircraft_count', ascending=False)

colors = [lpa_group_colors[g] for g in fleet_comp['lpa_group']]

fig = go.Figure(go.Bar(
    x=fleet_comp['lpa_group'],
    y=fleet_comp['aircraft_count'],
    marker_color=colors,
    text=fleet_comp['aircraft_count'].apply(lambda x: f'{x:,}'),
    textposition='outside',
    textfont=dict(size=10, color='black'),
    hovertemplate='<b>%{x}</b><br>Aircraft: %{y:,}<extra></extra>',
))

fig.update_layout(
    font=dict(color='black'),
    title=dict(
        text='<b>LPA* Fleet Composition by Aircraft Family (2022–2025, Eurocontrol Area)</b><br>'
             '<sup>Source: Author\'s own construction based on Eurocontrol OPDI data</sup>',
        x=0.5,
        font=dict(size=16, color='black')
    ),
    xaxis=dict(
        title='<b>AIRCRAFT FAMILY</b>',
        title_font=dict(size=11),
        tickfont=dict(size=10, color='black'),
        linecolor='black', linewidth=1.2
    ),
    yaxis=dict(
        title='<b>NUMBER OF UNIQUE AIRCRAFT</b>',
        title_font=dict(size=11),
        tickfont=dict(size=10, color='black'),
        linecolor='black', linewidth=1.2,
        showgrid=True, gridcolor='rgba(0,0,0,0.05)'
    ),
    paper_bgcolor='white',
    plot_bgcolor='white',
    height=550,
    width=1200,
    margin=dict(t=100, b=120, l=70, r=50),
    annotations=[dict(
        text="LPA* – Luxury Private Aviation",
        xref="paper", yref="paper",
        x=0.1, y=-0.25,
        xanchor="right", yanchor="bottom",
        showarrow=False,
        font=dict(size=9, color="grey"),
    )]
)

fig.show()
fig.write_image("Figure 5. LPA Fleet Composition by Aircraft Family (2022-2025, Eurocontrol Area).svg")

In [ ]:
df_flights_groups = pd.read_parquet(FULL_LPA_FILE, columns=['icao24', 'dof'])
df_flights_groups['dof'] = pd.to_datetime(df_flights_groups['dof'])
df_flights_groups['year'] = df_flights_groups['dof'].dt.year

df_merged_groups = df_flights_groups.merge(
    df_aircraft[df_aircraft['lpa_group'] != 'non_lpa'][['icao24', 'lpa_group']],
    on='icao24', how='inner'
)

all_years = sorted(df_merged_groups['year'].unique())
cumulative_rows = []
for group in lpa_group_colors.keys():
    df_group = df_merged_groups[df_merged_groups['lpa_group'] == group]
    seen = set()
    for year in all_years:
        new_aircraft = set(df_group[df_group['year'] == year]['icao24'].unique())
        seen.update(new_aircraft)
        cumulative_rows.append({'year': year, 'lpa_group': group, 'aircraft_count': len(seen)})

yearly_groups = pd.DataFrame(cumulative_rows)

fig = go.Figure()

text_positions = ['top right', 'bottom right', 'middle right', 'top right', 'bottom right', 'middle right', 'top right', 'bottom right', 'middle right', 'top right']

for i, (group, color) in enumerate(lpa_group_colors.items()):
    df_grp = yearly_groups[yearly_groups['lpa_group'] == group].sort_values('year')
    fig.add_trace(go.Scatter(
        x=df_grp['year'],
        y=df_grp['aircraft_count'],
        mode='lines+markers+text',
        name=group,
        line=dict(color=color, width=2),
        marker=dict(size=7, color=color, line=dict(color='#666666', width=0.8)),
        text=[group if j == len(df_grp) - 1 else '' for j in range(len(df_grp))],
        textposition=text_positions[i],
        textfont=dict(size=11, color='black'),
        hovertemplate=f'<b>{group}</b><br>Year: %{{x}}<br>Aircraft: %{{y}}<extra></extra>'
    ))

fig.update_layout(
    font=dict(color='black'),
    title=dict(
        text='<b>LPA* Cumulative Fleet Growth by Aircraft Family (2022–2025, Eurocontrol Area)</b><br>'
             '<sup>Source: Author\'s own construction based on Eurocontrol OPDI data</sup>',
        x=0.5, font=dict(size=16, color='black')
    ),
    xaxis=dict(
        title='<b>YEAR</b>',
        title_font=dict(size=11),
        tickmode='linear', dtick=1,
        tickfont=dict(size=10, color='black'),
        linecolor='black', linewidth=1.2,
        range=[2021.8, 2026.3]
    ),
    yaxis=dict(
        title='<b>CUMULATIVE UNIQUE AIRCRAFT</b>',
        title_font=dict(size=11),
        tickfont=dict(size=10, color='black'),
        linecolor='black', linewidth=1.2,
        showgrid=True, gridcolor='rgba(0,0,0,0.05)'
    ),
    showlegend=False,
    paper_bgcolor='white',
    plot_bgcolor='white',
    height=700,
    width=1000,
    margin=dict(t=100, b=80, l=50, r=50),
    hovermode='x unified',
    annotations=[dict(
        text="LPA* – Luxury Private Aviation",
        xref="paper", yref="paper",
        x=1, y=-0.12,
        xanchor="right", yanchor="bottom",
        showarrow=False,
        font=dict(size=9, color="grey"),
    )]
)

fig.show()
fig.write_image("Figure 6. LPA Cumulative Fleet Growth by Aircraft Family (2022-2025, Eurocontrol Area).svg")

###4.2.2. Fleet Ownership Analysis

In [ ]:
df_lpa_only = pq.read_table(FULL_LPA_FILE, columns=['icao24', 'ext_owner']).to_pandas().drop_duplicates(subset='icao24')

df_lpa_only['owner_known'] = (
    df_lpa_only['ext_owner'].notna() &
    (df_lpa_only['ext_owner'].str.strip() != '') &
    (df_lpa_only['ext_owner'].str.lower() != 'unknown')
)

print(df_lpa_only['owner_known'].value_counts())

owner_pie = df_lpa_only['owner_known'].value_counts().reset_index()
owner_pie.columns = ['known', 'count']
owner_pie['label'] = owner_pie['known'].map({True: 'Known Owner', False: 'Unknown Owner'})

df_known    = df_lpa_only[df_lpa_only['owner_known']].copy()
owner_fleet = df_known.groupby('ext_owner')['icao24'].nunique().reset_index()
owner_fleet.columns = ['ext_owner', 'fleet_count']

bins   = [0, 1, 2, 3, 4, 5, 10, 50, float('inf')]
labels = ['1', '2', '3', '4', '5', '6–10', '11–50', '50+']
owner_fleet['group'] = pd.cut(owner_fleet['fleet_count'], bins=bins, labels=labels)
fleet_dist = owner_fleet.groupby('group', observed=True)['ext_owner'].count().reset_index()
fleet_dist.columns = ['group', 'owner_count']
total = fleet_dist['owner_count'].sum()

fig = make_subplots(
    rows=1, cols=2,
    column_widths=[0.35, 0.65],
    specs=[[{"type": "pie"}, {"type": "bar"}]],
)
fig.add_trace(go.Pie(
    labels=owner_pie['label'],
    values=owner_pie['count'],
    showlegend=False,
    hole=0.4,
    marker=dict(colors=['#F9E076', '#B9B9B9']),
    texttemplate='<b>%{label}</b><br>%{percent}',
    textfont=dict(size=10, color='black'),
    insidetextorientation='horizontal',
    hovertemplate='<b>%{label}</b><br>Aircraft: %{value:,}<br>Share: %{percent}<extra></extra>',
    sort=False
), row=1, col=1)
fig.add_trace(go.Bar(
    x=fleet_dist['group'],
    y=fleet_dist['owner_count'],
    marker_color=['#F9E076', '#F8DA5E', '#F5CE3A', '#f27907', '#e8a86f', '#D4D4D4', '#B9B9B9', '#7A7A7A'],
    text=[f"{v:,}<br>({v/total*100:.1f}%)" for v in fleet_dist['owner_count']],
    textposition='outside',
    textfont=dict(size=9, color='black'),
    hovertemplate='<b>%{x}</b><br>Owners: %{y:,}<extra></extra>',
    showlegend=False
), row=1, col=2)
fig.update_layout(
    font=dict(color='black'),
    title=dict(
        text='<b>LPA* Fleet Ownership Analysis (2022–2025, Eurocontrol Area)</b><br>'
             '<sup>Source: Author\'s own construction based on Eurocontrol OPDI data</sup>',
        x=0.5, y=0.88, font=dict(size=16, color='black')
    ),
    paper_bgcolor='white', plot_bgcolor='white',
    height=550, width=1200,
    margin=dict(t=130, b=100, l=70, r=50),
    xaxis2=dict(
        title='<b>FLEET SIZE GROUP</b>', title_font=dict(size=11),
        tickfont=dict(size=10, color='black'), linecolor='black', linewidth=1.2
    ),
    yaxis2=dict(
        title='<b>NUMBER OF OWNERS</b>', title_font=dict(size=11),
        tickfont=dict(size=10, color='black'), linecolor='black', linewidth=1.2,
        showgrid=True, gridcolor='rgba(0,0,0,0.05)'
    ),
    annotations=[
        dict(text='<b>Owner Identifiability</b>', xref="paper", yref="paper",
             x=0.15, y=1.02, xanchor="center", yanchor="bottom",
             showarrow=False, font=dict(size=13, color="black")),
        dict(text='<b>Owner Distribution by Fleet Size</b>', xref="paper", yref="paper",
             x=0.70, y=1.02, xanchor="center", yanchor="bottom",
             showarrow=False, font=dict(size=13, color="black")),
        dict(text="LPA* – Luxury Private Aviation", xref="paper", yref="paper",
             x=1, y=-0.12, xanchor="right", yanchor="bottom",
             showarrow=False, font=dict(size=9, color="grey")),
    ]
)
fig.show()
fig.write_image("Figure 7. LPA Fleet Ownership Analysis (2022-2025, Eurocontrol Area).svg")

In [ ]:
df_known = df_lpa_only[df_lpa_only['owner_known']].copy()
top6_owners = (df_known.groupby('ext_owner')['icao24']
               .nunique()
               .sort_values(ascending=False)
               .head(6)
               .index.tolist())

df_top6  = df_known[df_known['ext_owner'].isin(top6_owners)]
stacked  = (df_top6.groupby('ext_owner')['icao24']
                   .nunique()
                   .reset_index()
                   .rename(columns={'icao24': 'aircraft_count'}))

df_flights_owners = pd.read_parquet(FULL_LPA_FILE, columns=['icao24', 'dof'])
df_flights_owners['dof']  = pd.to_datetime(df_flights_owners['dof'])
df_flights_owners['year'] = df_flights_owners['dof'].dt.year

df_merged = df_flights_owners.merge(
    df_known[df_known['ext_owner'].isin(top6_owners)][['icao24', 'ext_owner']],
    on='icao24', how='inner'
)

all_years = sorted(df_merged['year'].unique())
cumulative_rows = []
for owner in top6_owners:
    df_owner = df_merged[df_merged['ext_owner'] == owner]
    seen = set()
    for year in all_years:
        new_aircraft = set(df_owner[df_owner['year'] == year]['icao24'].unique())
        seen.update(new_aircraft)
        cumulative_rows.append({'year': year, 'ext_owner': owner, 'aircraft_count': len(seen)})
yearly = pd.DataFrame(cumulative_rows)

colors_line = ['#F8DA5E', '#f27907', '#B9B9B9', '#9E9E9E', '#F5CE3A', '#635a52']

fig = make_subplots(
    rows=1, cols=2,
    column_widths=[0.40, 0.60],
    specs=[[{"type": "bar"}, {"type": "scatter"}]],
)

owner_vals = {row['ext_owner']: row['aircraft_count'] for _, row in stacked.iterrows()}
fig.add_trace(go.Bar(
    name='Fleet Size',
    x=top6_owners,
    y=[owner_vals.get(o, 0) for o in top6_owners],
    marker_color='#F9E076',
    hovertemplate='<b>%{x}</b><br>Aircraft: %{y}<extra></extra>',
    showlegend=False
), row=1, col=1)

for i, owner in enumerate(top6_owners):
    df_owner = yearly[yearly['ext_owner'] == owner].sort_values('year')
    fig.add_trace(go.Scatter(
        x=df_owner['year'],
        y=df_owner['aircraft_count'],
        mode='lines+markers+text',
        name=owner,
        showlegend=False,
        line=dict(color=colors_line[i], width=2),
        marker=dict(size=7, color=colors_line[i]),
        text=[owner if j == len(df_owner) - 1 else '' for j in range(len(df_owner))],
        textposition='middle right',
        textfont=dict(size=10, color='black'),
        hovertemplate=f'<b>{owner}</b><br>Year: %{{x}}<br>Aircraft: %{{y}}<extra></extra>'
    ), row=1, col=2)

fig.update_layout(
    barmode='stack',
    font=dict(color='black'),
    title=dict(
        text='<b>Top 6 LPA* Owners — Fleet Size & Cumulative Growth (2022–2025, Eurocontrol Area)</b><br>'
             '<sup>Source: Author\'s own construction based on Eurocontrol OPDI data</sup>',
        x=0.5, y=0.89, font=dict(size=16, color='black')
    ),
    paper_bgcolor='white', plot_bgcolor='white',
    height=800, width=1200,
    margin=dict(t=130, b=150, l=40, r=50),
    legend=dict(orientation='h', yanchor='bottom', y=-0.75,
                xanchor='center', x=0.3, font=dict(size=9)),
    xaxis=dict(tickfont=dict(size=9, color='black'), linecolor='black', linewidth=1.2),
    yaxis=dict(
        title='<b>NUMBER OF UNIQUE AIRCRAFT</b>', title_font=dict(size=11),
        tickfont=dict(size=10, color='black'), linecolor='black', linewidth=1.2,
        showgrid=True, gridcolor='rgba(0,0,0,0.05)'
    ),
    xaxis2=dict(
        title='<b>YEAR</b>', title_font=dict(size=11),
        tickmode='linear', dtick=1,
        tickfont=dict(size=10, color='black'), linecolor='black', linewidth=1.2,
        range=[2021.8, 2026]
    ),
    yaxis2=dict(
        title='<b>CUMULATIVE UNIQUE AIRCRAFT</b>', title_font=dict(size=11),
        tickfont=dict(size=10, color='black'), linecolor='black', linewidth=1.2,
        showgrid=True, gridcolor='rgba(0,0,0,0.05)'
    ),
    annotations=[
        dict(text='<b>Fleet Size</b>', xref="paper", yref="paper",
             x=0.22, y=1.02, xanchor="center", yanchor="bottom",
             showarrow=False, font=dict(size=13, color="black")),
        dict(text='<b>Cumulative Fleet Growth by Year</b>', xref="paper", yref="paper",
             x=0.78, y=1.02, xanchor="center", yanchor="bottom",
             showarrow=False, font=dict(size=13, color="black")),
        dict(text="LPA* – Luxury Private Aviation", xref="paper", yref="paper",
             x=1, y=-0.42, xanchor="right", yanchor="bottom",
             showarrow=False, font=dict(size=9, color="grey")),
    ]
)
fig.show()
fig.write_image("Figure 8. Top 6 LPA Fleet Owners. Fleet Composition and Growth (2022-2025, Eurocontrol Area).svg")

In [ ]:
df_top6_groups = (pd.read_parquet(FULL_LPA_FILE, columns=['icao24', 'lpa_group'])
                    .drop_duplicates(subset='icao24'))
df_top6_groups = df_top6_groups.merge(df_top6[['icao24', 'ext_owner']], on='icao24', how='inner')

stacked_groups = (df_top6_groups.groupby(['ext_owner', 'lpa_group'])['icao24']
                                .nunique()
                                .reset_index()
                                .rename(columns={'icao24': 'aircraft_count'}))

fig = go.Figure()
for lpa_group, color in lpa_group_colors.items():
    df_grp = stacked_groups[stacked_groups['lpa_group'] == lpa_group]
    if df_grp.empty:
        continue
    owner_vals = {row['ext_owner']: row['aircraft_count'] for _, row in df_grp.iterrows()}
    fig.add_trace(go.Bar(
        name=lpa_group,
        x=top6_owners,
        y=[owner_vals.get(o, 0) for o in top6_owners],
        marker_color=color,
        hovertemplate='<b>%{x}</b><br>' + lpa_group + ': %{y}<extra></extra>',
    ))

fig.update_layout(
    barmode='stack',
    font=dict(color='black'),
    title=dict(
        text='<b>Top 6 LPA* Owners — Fleet Composition by Aircraft Family (2022–2025)</b><br>'
             '<sup>Source: Author\'s own construction based on Eurocontrol OPDI data</sup>',
        x=0.5, font=dict(size=14, color='black')
    ),
    xaxis=dict(tickfont=dict(size=9), linecolor='black', linewidth=1.2),
    yaxis=dict(
        title='<b>NUMBER OF UNIQUE AIRCRAFT</b>', title_font=dict(size=11),
        tickfont=dict(size=10), linecolor='black', linewidth=1.2,
        showgrid=True, gridcolor='rgba(0,0,0,0.05)'
    ),
    legend=dict(orientation='h', yanchor='top', y=-0.15, xanchor='center', x=0.5, font=dict(size=10)),
    paper_bgcolor='white', plot_bgcolor='white',
    height=500, width=900,
    margin=dict(t=100, b=120, l=60, r=50),
    annotations=[dict(
        text="LPA* – Luxury Private Aviation",
        xref="paper", yref="paper",
        x=1, y=-0.13, xanchor="right", yanchor="bottom",
        showarrow=False, font=dict(size=9, color="grey")
    )]
)
fig.show()
fig.write_image("Figure 9. Top 6 LPA Fleet Owners. Fleet Composition by Aircraft Family (2022-2025, Eurocontrol Area).svg")

###4.2.3. Fleet Behavior

I want to see the story behind the aicrafts and how they comprise the full traffic. Specifically, traffic share by groups of aicrafts that appear certain time in the dataset. I want to group them by appearance count and see their distribution among all unique fleet count.

In [ ]:
df = pd.read_parquet(FULL_LPA_FILE, columns=['icao24', 'id'])
aircraft_counts = df.groupby('icao24')['id'].count().reset_index()
aircraft_counts.columns = ['icao24', 'flight_count']
bins   = [0, 50, 200, 500, 1000, float('inf')]
labels = ['1–50 f/a', '51–200 f/a', '201–500 f/a', '501–1000 f/a', '1000+ f/a']
aircraft_counts['group'] = pd.cut(aircraft_counts['flight_count'], bins=bins, labels=labels)
pie_data = aircraft_counts.groupby('group', observed=True)['icao24'].count().reset_index()
pie_data.columns = ['group', 'aircraft_count']

stats = aircraft_counts['flight_count'].describe().round(1)

fig = make_subplots(
    rows=1, cols=2,
    column_widths=[0.30, 0.70],
    specs=[[{"type": "violin"}, {"type": "pie"}]]
)

fig.add_trace(go.Violin(
    y=aircraft_counts['flight_count'],
    name='Flight Appearances',
    box_visible=False,
    meanline_visible=True,
    fillcolor='#F9E076',
    line_color='#F9E076',
    opacity=0.8,
    points='outliers',
    marker=dict(color='#D3D3D3', size=3, opacity=0.5),
    hovertemplate='%{y} flights<extra></extra>'
), row=1, col=1)

fig.add_trace(go.Pie(
    labels=pie_data['group'],
    values=pie_data['aircraft_count'],
    hole=0.35,
    textposition='inside',
    insidetextorientation='horizontal',
    marker=dict(
        colors=['#D3D3D3', '#E8D98A', '#F9E076', '#e9dea2', '#ebe67b'],
    ),
    texttemplate='<b>%{label}</b><br>%{percent}',
    textfont=dict(size=10, color='black'),
    hovertemplate='<b>%{label} flights</b><br>Aircraft: %{value:,}<br>Fleet Share: %{percent}<extra></extra>',
    sort=False
), row=1, col=2)

fig.update_layout(
    font=dict(color='black'),
    title=dict(
        text='<b>LPA* Fleet Share by Flight Count (2022-2025, Eurocontrol Area) </b><br>'
             '<sup>Source: Author\'s own construction based on Eurocontrol OPDI data </sup>',
        x=0.5,
        font=dict(size=16, color='black')
    ),
    paper_bgcolor='white',
    plot_bgcolor='white',
    height=800,
    width=900,
    margin=dict(t=100, b=80, l=70, r=50),
    showlegend=False,
    yaxis=dict(
        title='<b>Number of Flights (per Aircraft)</b>',
        title_font=dict(size=11),
        showgrid=True,
        gridcolor='rgba(0,0,0,0.05)',
        linecolor='black',
        linewidth=1,
        tickfont=dict(size=10)
    ),
    xaxis=dict(showticklabels=False),
    annotations=[
        dict(
            text="LPA* – Luxury Private Aviation; f/a - flights per aircraft",
            xref="paper", yref="paper",
            x=0.9, y=-0.05,
            xanchor="right", yanchor="bottom",
            showarrow=False,
            font=dict(size=9, color="grey"),
        ),
        dict(
            text=(
                f"<b>Statistics</b><br>"
                f"Count: {int(stats['count']):,} total aircrafts<br>"
                f"Mean: {stats['mean']} f/a<br>"
                f"Median: {stats['50%']} f/a<br>"
                f"Q1: {stats['25%']} f/a<br>"
                f"Q3: {stats['75%']} f/a<br>"
            ),
            xref="paper", yref="paper",
            x=0.2, y=0.99,
            xanchor="left", yanchor="top",
            showarrow=False,
            font=dict(size=10, color="black"),
            bgcolor="rgba(245,245,245,0.8)",
            bordercolor="lightgrey",
            borderwidth=1,
            align="left"
        ),
        dict(
            text="<b>Distribution of Aircraft</b>",
            xref="paper", yref="paper",
            x=0.18, y=-0.08,
            xanchor="center", yanchor="top",
            showarrow=False,
            font=dict(size=11, color="black"),
        ),
        dict(
            text="<b> Total Fleet Share</b>",
            xref="paper", yref="paper",
            x=0.80, y=-0.08,
            xanchor="center", yanchor="top",
            showarrow=False,
            font=dict(size=11, color="black"),
        ),
    ]
)

fig.show()
fig.write_image("Figure 10. LPA Fleet Share by Flight Count (2022-2025, Eurocontrol Area).svg")

More than half of the total fleet count flew fewer than 50 times over the entire 2022–2025 period — these are low-utilization or sporadically active aircraft, most likely belonging to private owners who fly on an ad-hoc, personal basis. Only 13.3% of aircraft (501–1000 and 1000+ groups combined) flew frequently enough to be considered operationally active regulars, which are likely charter or fractional ownership aircraft operated by commercial operators on high-frequency schedules. The violin plot confirms this structural divide — the vast majority of aircraft cluster near zero, while a small number of extreme outliers reach 3,500+ flights over the period. This suggests the LPA fleet is highly heterogeneous: a large base of privately-owned, occasionally-used aircraft coexisting with a small but traffic-dominant core of commercially-operated, intensively-flown jets.

In [ ]:
cols = ['icao24', 'dof', 'adep', 'ades', 'first_seen', 'last_seen']
df_tat = pd.read_parquet(FULL_LPA_FILE, columns=cols)
df_tat['dof']        = pd.to_datetime(df_tat['dof'])
df_tat['first_seen'] = pd.to_datetime(df_tat['first_seen'])
df_tat['last_seen']  = pd.to_datetime(df_tat['last_seen'])

df_tat = df_tat[
    (df_tat['adep'] != 'Unknown/Outside Wider Europe') &
    (df_tat['ades'] != 'Unknown/Outside Wider Europe')
].copy()

df_tat = df_tat.sort_values(['icao24', 'first_seen']).reset_index(drop=True)
df_tat['next_first_seen'] = df_tat.groupby('icao24')['first_seen'].shift(-1)
df_tat['next_adep']       = df_tat.groupby('icao24')['adep'].shift(-1)
df_tat_valid = df_tat[
    (df_tat['ades'] == df_tat['next_adep']) &
    (df_tat['next_first_seen'].notna())
].copy()

df_tat_valid['turnaround_hours'] = (
    df_tat_valid['next_first_seen'] - df_tat_valid['last_seen']
).dt.total_seconds() / 3600

df_tat_valid = df_tat_valid[
    (df_tat_valid['turnaround_hours'] >= 0.5) &
    (df_tat_valid['turnaround_hours'] <= 720)
].copy()


bins   = [0, 2, 6, 24, 72, 168, 720]
labels = ['<2h\n(Quick Turn)', '2–6h\n(Day Trip)', '6–24h\n(Overnight)',
          '1–3 Days', '3–7 Days', '7–30 Days']

df_tat_valid['tat_group'] = pd.cut(
    df_tat_valid['turnaround_hours'], bins=bins, labels=labels
)

tat_counts = (df_tat_valid.groupby('tat_group', observed=True).size()
                           .reset_index(name='count'))
tat_counts['pct'] = (tat_counts['count'] / tat_counts['count'].sum() * 100).round(1)

fig = make_subplots(rows=1, cols=2,
    column_widths=[0.4, 0.6],
    specs=[[{'type': 'histogram'}, {'type': 'bar'}]])

fig.add_trace(go.Histogram(
    x=df_tat_valid['turnaround_hours'],
    nbinsx=100,
    marker_color='#F9E076',
    marker_line=dict(color='white', width=0.3),
    hovertemplate='Turnaround: %{x:.1f}h<br>Count: %{y:,}<extra></extra>',
    name='Turnaround'
), row=1, col=1)

fig.add_trace(go.Bar(
    x=tat_counts['tat_group'],
    y=tat_counts['count'],
    marker_color=['#a6a6a6','#cccccc','#fae68e','#f8db5d','#f5cb14','#b59508'],
    text=tat_counts.apply(lambda r: f"{r['count']:,}<br>({r['pct']}%)", axis=1),
    textposition='outside',
    textfont=dict(size=9),
    hovertemplate='<b>%{x}</b><br>Count: %{y:,}<extra></extra>',
    showlegend=False
), row=1, col=2)

fig.update_layout(
    font=dict(color='black'),
    title=dict(
        text='<b>LPA* Aircraft Turnaround Time Analysis (2022–2025, Eurocontrol Area)</b><br>'
             '<sup>Time between landing and next departure from same airport | '
             'Source: Author\'s own analysis based on Eurocontrol OPDI data</sup>',
        x=0.5, font=dict(size=15, color='black')
    ),
    xaxis=dict(title='Turnaround Time (hours)', tickfont=dict(size=9), title_font=dict(size=10),range=[0, 400]),
    yaxis=dict(title='Number of Flights', tickfont=dict(size=9), title_font=dict(size=10)),
    xaxis2=dict(tickfont=dict(size=8), title_font=dict(size=9)),
    yaxis2=dict(title='Count', tickfont=dict(size=9), title_font=dict(size=10)),
    paper_bgcolor='white',
    plot_bgcolor='white',
    template='plotly_white',
    height=500,
    showlegend=False,
    margin=dict(t=100, l=60, r=60, b=80),
    annotations=[
        dict(text='<b>Turnaround Time Distribution</b>',
             xref='paper', yref='paper', x=0.25, y=-0.18,
             xanchor='center', showarrow=False, font=dict(size=11)),
        dict(text='<b>Turnaround Categories</b>',
             xref='paper', yref='paper', x=0.82, y=-0.18,
             xanchor='center', showarrow=False, font=dict(size=11)),
        dict(text="LPA* – Luxury Private Aviation",
             xref="paper", yref="paper",
             x=1.0, y=-0.1,
             xanchor="right", yanchor="bottom",
             showarrow=False, font=dict(size=9, color="grey"))
    ]
)
fig.show()
fig.write_image("Figure 11. Aircraft Turnaround Time Analysis (2022-2025, Eurocontrol Area).svg")
print(f"Valid turnaround records: {len(df_tat_valid):,}")
print(f"\nTurnaround Time (hours):")
print(df_tat_valid['turnaround_hours'].describe().round(2))
print("\n Turnaround Category Breakdown")
print(tat_counts.to_string(index=False))

Turnaround time analysis - how long does an aircraft stay at destination before returning (last_seen at ades vs next first_seen at adep)?753,478 valid records reveals a median ground time of 12.86 hours, though the distribution is heavily right-skewed. 30.9% of all turnarounds occur within 2 hours — consistent with the earlier finding that a significant share of LPA traffic comprises charter flight operations or repositioning flights.

## 4.3 Wealthy Few Mobilities

###4.3.1. Destinations

In [ ]:
df_country = pd.read_parquet(FULL_LPA_FILE, columns=['ades_country_name'])

ades_counts = (df_country[df_country['ades_country_name'] != 'Unknown/Outside Wider Europe']
               ['ades_country_name']
               .value_counts()
               .head(15)
               .reset_index()
               .rename(columns={'ades_country_name': 'country', 'count': 'flights'}))

fig = go.Figure()
fig.add_trace(go.Bar(
    x=ades_counts['flights'],
    y=ades_counts['country'],
    orientation='h',
    marker_color='#F9E076',
    marker_line=dict(width=0),
    text=[f"{v:,}" for v in ades_counts['flights']],
    textposition='outside',
    textfont=dict(size=9, color='black'),
    hovertemplate='<b>%{y}</b><br>Flights: %{x:,}<extra></extra>',
))

fig.update_layout(
    font=dict(color='black'),
    title=dict(
        text='<b> Top 15 LPA* Destination Countries (2022–2025, Eurocontrol Area)</b><br>'
             '<sup>Source: Author\'s own analysis based on Eurocontrol OPDI data</sup>',
        x=0.5, font=dict(size=15, color='black')
    ),
    xaxis=dict(title='Total Arrivals', showgrid=True, gridcolor='rgba(0,0,0,0.05)',
               linecolor='black', linewidth=1, tickfont=dict(size=9)),
    yaxis=dict(categoryorder='total ascending', linecolor='black', linewidth=1, tickfont=dict(size=10)),
    paper_bgcolor='white', plot_bgcolor='white',
    height=500, width=800,
    margin=dict(t=100, b=60, l=150, r=80),
    showlegend=False,
    annotations=[dict(
        text="LPA* – Luxury Private Aviation",
        xref="paper", yref="paper",
        x=1, y=0, xanchor="right", yanchor="bottom",
        showarrow=False, font=dict(size=9, color="grey")
    )]
)
fig.show()
fig.write_image("Figure 12. Top 15 LPA Destination Countries (2022-2025, Eurocontrol Area).svg")


In [ ]:
def plot_lpa_arrivals_map():
    df = pd.read_parquet(FULL_LPA_FILE, columns=[
        'ades', 'ades_name', 'ades_latitude_deg', 'ades_longitude_deg'
    ])
    map_data = (df.dropna(subset=['ades_latitude_deg', 'ades_longitude_deg'])
                  .groupby(['ades', 'ades_name', 'ades_latitude_deg', 'ades_longitude_deg'])
                  .size()
                  .reset_index(name='Total_Arrivals'))

    fig = px.scatter_mapbox(
        map_data,
        lat="ades_latitude_deg", lon="ades_longitude_deg",
        size="Total_Arrivals",
        color="Total_Arrivals",
        color_continuous_scale=['#B9B9B9', '#FFBF00'],
        size_max=35,
        zoom=3.5,
        center={"lat": 48.0, "lon": 10.0},
        hover_name="ades_name",
        hover_data={"ades": True, "Total_Arrivals": True,
                    "ades_latitude_deg": False, "ades_longitude_deg": False},
        mapbox_style="carto-positron"
    )

    fig.update_layout(
        title=dict(
            text="<b>Geospatial Concentration of Total LPA* Flights Landed (2022–2025, Eurocontrol Area)</b><br>"
                 "<sup>Source: Author's own analysis based on Eurocontrol OPDI data</sup>",
            x=0.5, font=dict(color="black", size=14)
        ),
        width=1000, height=800,
        paper_bgcolor='white',
        font=dict(color="black", size=10),
        margin=dict(t=100, l=20, r=20, b=20),
        annotations=[dict(
            text="LPA* – Luxury Private Aviation",
            xref="paper", yref="paper",
            x=0.01, y=0.01, xanchor="left", yanchor="bottom",
            showarrow=False, font=dict(size=8, color="grey")
        )],
        coloraxis_colorbar=dict(
            title="<b>TOTAL ARRIVALS</b>",
            title_font=dict(size=12),
            thickness=15, len=0.7,
            xanchor="left", x=0.02
        )
    )
    fig.show()
    return fig

if __name__ == "__main__":
    generated_fig = plot_lpa_arrivals_map()
    generated_fig.write_image("Figure 13. Geospatial Concentration of Total LPA Flights Landed (2022-2025, Eurocontrol Area).svg")


Despite the UK and Germany generating high total flight volumes at the country level, their traffic is spread across multiple aerodromes — visible as many medium-sized bubbles scattered across both countries. France, by contrast, shows a sharper concentration pattern where a small number of aerodromes absorb the bulk of movements, suggesting French LPA demand funnels through established prestige hubs rather than dispersing regionally. Similarly, Geneva and Milan stand out as disproportionately large bubbles relative to their national context — neither Switzerland nor northern Italy generates massive country-level totals, yet these two cities rival or exceed many capital airports, reflecting their role as financial and luxury lifestyle hubs that attract high-net-worth traffic independently of broader national demand.

In [ ]:
def plot_municipality_weekly_bubbles_combined():
    df = pd.read_parquet(FULL_LPA_FILE, columns=['dof', 'ades_municipality'])
    df['dof'] = pd.to_datetime(df['dof'])
    df = df[df['ades_municipality'] != 'Unknown/Outside Wider Europe'].copy()
    df['week']    = df['dof'].dt.isocalendar().week.astype(int)
    df['quarter'] = 'Q' + df['dof'].dt.quarter.astype(str)

    weekly_counts = (df.groupby(['week', 'ades_municipality'])
                       .size()
                       .reset_index(name='count'))

    week_to_quarter = (df[['week','quarter']]
                         .drop_duplicates()
                         .sort_values('week')
                         .drop_duplicates('week'))

    ranked = (weekly_counts.sort_values(['week', 'count'], ascending=[True, False])
                            .groupby('week')
                            .head(2)
                            .copy())
    ranked['rank'] = ranked.groupby('week')['count'].rank(ascending=False, method='first').astype(int)
    ranked = ranked.merge(week_to_quarter, on='week', how='left')

    unique_munis = ranked['ades_municipality'].unique()
    palette = [
        '#F9E076','#F4B942','#F97B3D','#B9B9B9','#7D6608',
        '#E8D5A3','#D4A017','#C68642','#8B6914','#FFD700',
        '#FFA500','#FF8C00','#DAA520','#B8860B','#CD853F'
    ]
    color_map = {name: palette[i % len(palette)] for i, name in enumerate(unique_munis)}

    # rank styling
    rank_symbol  = {1: 'circle',         2: 'circle-open'}
    rank_label   = {1: '1st',            2: '2nd'}
    rank_opacity = {1: 1.0,              2: 0.75}

    fig = go.Figure()
    legend_added = set()

    for _, row in ranked.iterrows():
        rank     = row['rank']
        muni     = row['ades_municipality']
        show_leg = rank not in legend_added
        if show_leg:
            legend_added.add(rank)

        fig.add_trace(go.Scatter(
            x=[row['week']],
            y=[muni],
            mode='markers',
            marker=dict(
                size=row['count'] / ranked['count'].max() * 55 + 10,
                color=color_map[muni],
                symbol=rank_symbol[rank],
                opacity=rank_opacity[rank],
                line=dict(color='white' if rank == 1 else color_map[muni], width=1.5)
            ),
            name=rank_label[rank],
            legendgroup=rank_label[rank],
            showlegend=show_leg,
            hovertemplate=(
                f"<b>Week {int(row['week'])} ({row['quarter']})</b><br>"
                f"Rank: {rank_label[rank]}<br>"
                f"Municipality: {muni}<br>"
                f"Flights: {int(row['count'])}<extra></extra>"
            )
        ))

    quarter_starts = {'Q1': 1, 'Q2': 14, 'Q3': 27, 'Q4': 40}
    for q, w in quarter_starts.items():
        fig.add_vline(x=w, line_dash='dash', line_color='gray', line_width=0.8, opacity=0.5)
        fig.add_annotation(
            x=w + 3, y=len(unique_munis) - 0.3,
            text=f'<b>{q}</b>', showarrow=False,
            font=dict(size=11, color='gray')
        )

    fig.update_layout(
        font=dict(color='black'),
        width=1200,
        title=dict(
            text='<b>Top 2 LPA* Destination Municipalities by Week of The Year (2022–2025, Eurocontrol Area)</b><br>'
                 '<sup>Source: Author\'s own analysis based on Eurocontrol OPDI data</sup>',
            x=0.5, font=dict(size=15)
        ),
        xaxis=dict(
            title='Week of Year', tickmode='linear', dtick=1,
            range=[0, 53], showgrid=True, gridcolor='#f0f0f0'
        ),
        yaxis=dict(title='Municipality', showgrid=True, gridcolor='#f0f0f0'),
        paper_bgcolor='white', plot_bgcolor='white',
        template='plotly_white', height=550,
        margin=dict(t=100, l=150, r=50, b=60),
        legend=dict(title='Rank', orientation='v', x=1.02, y=1),
        annotations=[dict(
            text="LPA* – Luxury Private Aviation",
            xref="paper", yref="paper",
            x=1, y=0, xanchor="right", yanchor="bottom",
            showarrow=False, font=dict(size=9, color="grey")
        )]
    )
    fig.show()
    return fig

if __name__ == "__main__":
    generated_fig = plot_municipality_weekly_bubbles_combined()
    generated_fig.write_image("Figure 14. Top 2 LPA Destination Municipalities by Week of The Year (2022–2025, Eurocontrol Area).svg")

In [ ]:
df_ades = pd.read_parquet(FULL_LPA_FILE, columns=['ades', 'ades_name'])

ades_counts = (df_ades[df_ades['ades'] != 'Unknown/Outside Wider Europe']
               .assign(label=df_ades['ades'] + ' – ' + df_ades['ades_name'])
               ['label']
               .value_counts()
               .head(15)
               .reset_index()
               .rename(columns={'label': 'airport', 'count': 'flights'}))

fig = go.Figure()
fig.add_trace(go.Bar(
    x=ades_counts['flights'],
    y=ades_counts['airport'],
    orientation='h',
    marker_color='#F9E076',
    marker_line=dict(width=0),
    text=[f"{v:,}" for v in ades_counts['flights']],
    textposition='outside',
    textfont=dict(size=9, color='black'),
    hovertemplate='<b>%{y}</b><br>Arrivals: %{x:,}<extra></extra>',
))

fig.update_layout(
    font=dict(color='black'),
    title=dict(
        text='<b>Top 15 LPA* Destination Airports (2022–2025, Eurocontrol Area)</b><br>'
             '<sup>Source: Author\'s own analysis based on Eurocontrol OPDI data</sup>',
        x=0.5, font=dict(size=15, color='black')
    ),
    xaxis=dict(title='Total Arrivals', showgrid=True, gridcolor='rgba(0,0,0,0.05)',
               linecolor='black', linewidth=1, tickfont=dict(size=9)),
    yaxis=dict(categoryorder='total ascending', tickfont=dict(size=10)),
    paper_bgcolor='white', plot_bgcolor='white',
    height=500, width=800,
    margin=dict(t=100, b=60, l=200, r=80),
    showlegend=False,
    annotations=[dict(
        text="LPA* – Luxury Private Aviation",
        xref="paper", yref="paper",
        x=1, y=0, xanchor="right", yanchor="bottom",
        showarrow=False, font=dict(size=9, color="grey")
    )]
)
fig.show()
fig.write_image("Figure 15. Top 15 LPA* Destination Airports (2022–2025, Eurocontrol Area).svg")


In [ ]:
def compute_growth(df, group_col, excl_top_n=6):

    d = df.copy()
    d = d[d[group_col] != 'Unknown/Outside Wider Europe']
    d['year'] = pd.to_datetime(d['dof']).dt.year

    top_n_exclude = set(d[group_col].value_counts().head(excl_top_n).index)

    yearly = (d.groupby(['year', group_col])
               .size()
               .reset_index(name='flights'))
    yearly = yearly[~yearly[group_col].isin(top_n_exclude)]

    pivot = yearly.pivot(index=group_col, columns='year', values='flights').fillna(0)
    pivot = pivot[pivot[2022] > 0]
    pivot['growth_abs'] = pivot[2025] - pivot[2022]
    pivot['cagr_pct']   = ((pivot[2025] / pivot[2022]) ** (1/3) - 1) * 100
    pivot.columns.name  = None
    return pivot.reset_index(), top_n_exclude


def plot_growth(pivot, group_col, metric='cagr_pct', top_n=10, title=None, width=1200):
    top = pivot.nlargest(top_n, metric).copy()

    if metric == 'cagr_pct':
        x_vals    = top['cagr_pct']
        x_title   = 'CAGR† (% per year)'
        text_vals = [f"+{v:.1f}%" for v in x_vals]
        secondary = 'growth_abs'
        sec_label = 'Absolute'
        sec_fmt   = '.0f'
    else:
        x_vals    = top['growth_abs']
        x_title   = 'Absolute Flight Growth (2022–2025)'
        text_vals = [f"+{v:.0f}" for v in x_vals]
        secondary = 'cagr_pct'
        sec_label = 'CAGR'
        sec_fmt   = '.1f'

    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=x_vals,
        y=top[group_col],
        orientation='h',
        marker_color='#F9E076',
        marker_line=dict(width=0),
        text=text_vals,
        textposition='outside',
        textfont=dict(size=9, color='black'),
        customdata=top[[secondary, 2022, 2023, 2024, 2025]].values,
        hovertemplate=(
            f'<b>%{'{y}'}</b><br>'
            f'{x_title}: %{'{x:'}{sec_fmt}{'}'}<br>'
            f'{sec_label}: %{'{customdata[0]:'}{sec_fmt}{'}'}<br>'
            f'2022: %{'{customdata[1]:.0f}'} → 2023: %{'{customdata[2]:.0f}'} → '
            f'2024: %{'{customdata[3]:.0f}'} → 2025: %{'{customdata[4]:.0f}'}<extra></extra>'
        )
    ))

    annotations = [
        dict(text="LPA* – Luxury Private Aviation",
             xref="paper", yref="paper",
             x=1, y=0, xanchor="right", yanchor="bottom",
             showarrow=False, font=dict(size=9, color="grey")),
    ]
    if metric == 'cagr_pct':
        annotations.append(
            dict(text="CAGR† – Compound Annual Growth Rate: annualised average growth rate over the 2022–2025 period",
                 xref="paper", yref="paper",
                 x=0.6, y=-0.12, xanchor="left", yanchor="top",
                 showarrow=False, font=dict(size=9, color="grey"))
        )

    fig.update_layout(
        font=dict(color='black'),
        title=dict(text=title or f'<b>Top {top_n} by {metric}</b>', x=0.5, font=dict(size=14, color='black')),
        xaxis=dict(title=x_title, showgrid=True, gridcolor='rgba(0,0,0,0.05)'),
        yaxis=dict(categoryorder='total ascending'),
        paper_bgcolor='white', plot_bgcolor='white',
        height=450, width=1200,
        margin=dict(t=100, b=80, l=50, r=50),
        annotations=annotations
    )
    fig.show()
    return fig

In [ ]:
df_growth = pd.read_parquet(FULL_LPA_FILE, columns=['dof', 'ades_country_name', 'ades_municipality', 'ades_name'])
df_growth['dof'] = pd.to_datetime(df_growth['dof'])

In [ ]:
pivot_countries, _ = compute_growth(df_growth, 'ades_country_name', excl_top_n=6)
pivot_munis,     _ = compute_growth(df_growth, 'ades_municipality',  excl_top_n=6)
pivot_airports,  _ = compute_growth(df_growth, 'ades_name',               excl_top_n=6)

In [ ]:
pivot_countries, _ = compute_growth(df_growth, 'ades_country_name', excl_top_n=6)
generated_fig = plot_growth(pivot_countries, 'ades_country_name', metric='cagr_pct', top_n=10,
            title='<b>Fastest Growing LPA* Country Destinations by CAGR (2022–2025, excl. Top 6)</b><br>'
                  '<sup>Source: Author\'s own analysis based on Eurocontrol OPDI data</sup>')
generated_fig.write_image("Figure 16. Fastest Growing LPA* Country Destinations by CAGR (2022–2025, Eurocontrol Area).svg")

In [ ]:
generated_fig = plot_growth(pivot_countries, 'ades_country_name', metric='growth_abs', top_n=10,
            title='<b>Top LPA* Countries by Average Yearly Volume Increase (2022–2025, excl. Top 6)</b><br>'
                  '<sup>Source: Author\'s own analysis based on Eurocontrol OPDI data</sup>')
generated_fig.write_image("Figure 17. Top LPA* Countries by Average Yearly Volume Increase (2022–2025, Eurocontrol Area).svg")

In [ ]:
generated_fig = plot_growth(pivot_munis, 'ades_municipality', metric='cagr_pct', top_n=10,
            title='<b>Fastest Growing LPA* Municipalities by CAGR (2022–2025, excl. Top 6)</b><br>'
                  '<sup>Source: Author\'s own analysis based on Eurocontrol OPDI data</sup>')
generated_fig.write_image("Figure 18. Fastest Growing LPA* Municipalities by CAGR (2022–2025, Eurocontrol Area).svg")

In [ ]:
generated_fig = plot_growth(pivot_munis, 'ades_municipality', metric='growth_abs', top_n=10,
            title='<b>Top LPA* Municipalities by Average Yearly Volume Increase (2022–2025, excl. Top 6)</b><br>'
                  '<sup>Source: Author\'s own analysis based on Eurocontrol OPDI data</sup>')
generated_fig.write_image("Figure 19. Top LPA* Municipalities by Average Yearly Volume Increase (2022–2025, Eurocontrol Area).svg")

In [ ]:
generated_fig = plot_growth(pivot_airports, 'ades_name', metric='cagr_pct', top_n=10,
            title='<b>Fastest Growing LPA* Airports by CAGR (2022–2025, excl. Top 6)</b><br>'
                  '<sup>Source: Author\'s own analysis based on Eurocontrol OPDI data</sup>')
generated_fig.write_image("Figure 20. Fastest Growing LPA* Airports by CAGR (2022–2025, Eurocontrol Area).svg")

In [ ]:
generated_fig = plot_growth(pivot_airports, 'ades_name', metric='growth_abs', top_n=10,
            title='<b>Top Growing LPA* Airports by Average Yearly Volume Increase (2022–2025, excl. Top 6)</b><br>'
                  '<sup>Source: Author\'s own analysis based on Eurocontrol OPDI data</sup>')
generated_fig.write_image("Figure 21. Top Growing LPA* Airports by Average Yearly Volume Increase (2022–2025, Eurocontrol Area).svg")

###4.3.2 Routes

In [ ]:
def plot_lpa_country_pairs():
    df = pd.read_parquet(FULL_LPA_FILE, columns=['adep_country_name', 'ades_country_name'])
    df = df[
    (df['adep_country_name'] != 'Unknown/Outside Wider Europe') &
    (df['ades_country_name'] != 'Unknown/Outside Wider Europe')
].copy()

    df['pair'] = df.apply(
        lambda row: ' ↔ '.join(sorted([row['adep_country_name'], row['ades_country_name']])),
        axis=1
    )

    pair_counts = (df.groupby('pair')
                     .size()
                     .reset_index(name='flights')
                     .sort_values('flights', ascending=True)
                     .tail(20))

    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=pair_counts['flights'],
        y=pair_counts['pair'],
        orientation='h',
        marker_color='#F9E076',
        marker_line=dict(width=0),
        text=[f"{v:,}" for v in pair_counts['flights']],
        textposition='outside',
        textfont=dict(size=9, color='black'),
        hovertemplate='<b>%{y}</b><br>Flights: %{x:,}<extra></extra>',
    ))

    fig.update_layout(
        font=dict(color='black'),
        title=dict(
            text='<b>Top 20 LPA* Bidirectional Country Pairs (2022–2025, Eurocontrol Area)</b><br>'
                 '<sup>Source: Author\'s own analysis based on Eurocontrol OPDI data</sup>',
            x=0.5, font=dict(size=15, color='black')
        ),
        paper_bgcolor='white', plot_bgcolor='white',
        height=700, width=1200,
        margin=dict(t=100, b=80, l=250, r=120),
        showlegend=False,
        xaxis=dict(title='<b>Number of Flights</b>', title_font=dict(size=11),
                   showgrid=True, gridcolor='rgba(0,0,0,0.05)', tickfont=dict(size=9)),
        yaxis=dict(tickfont=dict(size=9)),
        annotations=[dict(
            text="LPA* – Luxury Private Aviation",
            xref="paper", yref="paper",
            x=1, y=0, xanchor="right", yanchor="bottom",
            showarrow=False, font=dict(size=9, color="grey")
        )]
    )
    fig.show()
    return fig

generated_fig = plot_lpa_country_pairs()
generated_fig.write_image("Figure 22. Top 20 LPA* Bidirectional Country Pairs (2022–2025, Eurocontrol Area).svg")

In [ ]:
def plot_lpa_airport_pairs():
    df = pd.read_parquet(FULL_LPA_FILE, columns=[
        'adep_name', 'adep_municipality',
        'ades_name', 'ades_municipality'
    ])
    df_verified = df[
        (df['adep_name'] != "Unknown/Outside Wider Europe") &
        (df['ades_name'] != "Unknown/Outside Wider Europe")
    ].copy()
    df_verified['DEPARTURE HUB'] = df_verified['adep_name'] + " (" + df_verified['adep_municipality'] + ")"
    df_verified['ARRIVAL HUB']   = df_verified['ades_name'] + " (" + df_verified['ades_municipality'] + ")"
    df_verified['pair'] = df_verified.apply(
        lambda row: ' ↔ '.join(sorted([row['DEPARTURE HUB'], row['ARRIVAL HUB']])),
        axis=1
    )
    pair_counts = (df_verified.groupby('pair')
                   .size()
                   .reset_index(name='flights')
                   .sort_values('flights', ascending=True)
                   .tail(20))
    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=pair_counts['flights'],
        y=pair_counts['pair'],
        orientation='h',
        marker_color='#F9E076',
        marker_line=dict(width=0),
        text=[f"{v:,}" for v in pair_counts['flights']],
        textposition='outside',
        textfont=dict(size=9, color='black'),
        hovertemplate='<b>%{y}</b><br>Flights: %{x:,}<extra></extra>',
    ))
    fig.update_layout(
        font=dict(color='black'),
        title=dict(
            text='<b>Top 20 LPA* Bidirectional Airport Pairs (2022–2025, Eurocontrol Area)</b><br>'
                 '<sup>Source: Author\'s own analysis based on Eurocontrol OPDI data</sup>',
            x=0.5,
            font=dict(size=15, color='black')
        ),
        paper_bgcolor='white',
        plot_bgcolor='white',
        height=700,
        width=1200,
        margin=dict(t=100, b=80, l=350, r=120),
        showlegend=False,
        xaxis=dict(
            title='<b>Number of Flights</b>',
            title_font=dict(size=11),
            showgrid=True,
            gridcolor='rgba(0,0,0,0.05)',
            linecolor='black',
            linewidth=1,
            tickfont=dict(size=9)
        ),
        yaxis=dict(
            linecolor='black',
            linewidth=1,
            tickfont=dict(size=9)
        ),
        annotations=[dict(
            text="LPA* – Luxury Private Aviation",
            xref="paper", yref="paper",
            x=1, y=0,
            xanchor="right", yanchor="bottom",
            showarrow=False,
            font=dict(size=9, color="grey"),
        )]
    )
    fig.show()
    return fig

generated_fig = plot_lpa_airport_pairs()
generated_fig.write_image("Figure 23. Top 20 LPA* Bidirectional Airport Pairs (2022–2025, Eurocontrol Area).svg")

In [ ]:
df_growth_country_pairs = pd.read_parquet(FULL_LPA_FILE, columns=['dof', 'adep_country_name', 'ades_country_name'])
df_growth_country_pairs['dof'] = pd.to_datetime(df_growth_country_pairs['dof'])
df_growth_country_pairs = df_growth_country_pairs[
    (df_growth_country_pairs['adep_country_name'] != 'Unknown/Outside Wider Europe') &
    (df_growth_country_pairs['ades_country_name'] != 'Unknown/Outside Wider Europe')
].copy()

df_growth_country_pairs['pair'] = df_growth_country_pairs.apply(
    lambda row: ' ↔ '.join(sorted([row['adep_country_name'], row['ades_country_name']])),
    axis=1
)
df_growth_country_pairs['ades_country_name'] = df_growth_country_pairs['pair']

pivot_country_pairs, _ = compute_growth(df_growth_country_pairs, 'ades_country_name', excl_top_n=6)

generated_fig = plot_growth(pivot_country_pairs, 'ades_country_name', metric='cagr_pct', top_n=10,
            title='<b>Fastest Growing LPA* Country Pairs by CAGR (2022–2025, Eurocontrol Area, excl. Top 6)</b><br>'
                  '<sup>Source: Author\'s own analysis based on Eurocontrol OPDI data</sup>',
            width=1300)
generated_fig.write_image("Figure 24. Fastest Growing LPA* Country Pairs by CAGR (2022–2025, Eurocontrol Area, excl. Top 6).svg")

In [ ]:
generated_fig = plot_growth(pivot_country_pairs, 'ades_country_name', metric='growth_abs', top_n=10,
            title='<b>Top LPA* Country Pairs by Volume Growth (2022–2025, Eurocontrol Area, excl. Top 6)</b><br>'
                  '<sup>Source: Author\'s own analysis based on Eurocontrol OPDI data</sup>',
            width=1300)
generated_fig.write_image("Figure 25. Top LPA* Country Pairs by Volume Growth (2022–2025, Eurocontrol Area, excl. Top 6).svg")

In [ ]:
df_growth_pairs = pd.read_parquet(FULL_LPA_FILE, columns=['dof', 'adep_name', 'ades_name'])
df_growth_pairs['dof'] = pd.to_datetime(df_growth_pairs['dof'])
df_growth_pairs = df_growth_pairs.dropna(subset=['adep_name', 'ades_name']).copy()
df_growth_pairs = df_growth_pairs[
    (df_growth_pairs['adep_name'] != 'Unknown/Outside Wider Europe') &
    (df_growth_pairs['ades_name'] != 'Unknown/Outside Wider Europe')
].copy()

df_growth_pairs['pair'] = df_growth_pairs.apply(
    lambda row: ' ↔ '.join(sorted([row['adep_name'], row['ades_name']])),
    axis=1
)
df_growth_pairs['ades_country_name'] = df_growth_pairs['pair']

pivot_pairs, _ = compute_growth(df_growth_pairs, 'ades_country_name', excl_top_n=6)

generated_fig = plot_growth(pivot_pairs, 'ades_country_name', metric='cagr_pct', top_n=10,
            title='<b>Fastest Growing LPA* Airport Pairs by CAGR (2022–2025, Eurocontrol Area, excl. Top 6)</b><br>'
                  '<sup>Source: Author\'s own analysis based on Eurocontrol OPDI data</sup>',
            width=1600)
generated_fig.write_image("Figure 26. Fastest Growing LPA* Airport Pairs by CAGR (2022–2025, Eurocontrol Area, excl. Top 6).svg")

In [ ]:
generated_fig = plot_growth(pivot_pairs, 'ades_country_name', metric='growth_abs', top_n=10,
            title='<b> Top LPA* Airport Pairs by Average Yearly Volume Increase (2022–2025, Eurocontrol Area, excl. Top 6)</b><br>'
                  '<sup>Source: Author\'s own analysis based on Eurocontrol OPDI data</sup>',
            width=1600)
generated_fig.write_image("Figure 27. Top LPA* Airport Pairs by Average Yearly Volume Increase (2022–2025, Eurocontrol Area, excl. Top 6).svg")

##4.4. Exploratory Example of Using LPA Traffic To Predict Segment Turnover

!cautioin! It is best advised to compare LPA traffic data with values that actually represents customer segment of LPA users - e.g. luxury retail companies - to analyze relationship with LPA arrivals to a certain desination and their VIP customer performance metrics. As such data is hard to acquire for this independent project, I will use just an exploratory approach with publicly available data that is most relevant. Additionally, the selected and acquired data has few data points, but neverthless, serves as the exploratory framework practice.

In [ ]:
cols = ['dof', 'ades_country_name', 'adep_country_name']
df_lpa_full = pd.read_parquet(FULL_LPA_FILE, columns=cols)
df_lpa_full['dof'] = pd.to_datetime(df_lpa_full['dof'])

mask = (
    (df_lpa_full['ades_country_name'] == 'Italy') &
    (df_lpa_full['adep_country_name'] != 'Italy')
)
daily_non_d = df_lpa_full[mask].groupby('dof').size()

full_range = pd.date_range(start='2022-01-01', end='2025-12-31')
non_d_italy_lpa_arrivals = daily_non_d.reindex(full_range, fill_value=0).astype(float)

gap_dates = pd.to_datetime(possible_data_gaps)
non_d_italy_lpa_arrivals.loc[non_d_italy_lpa_arrivals.index.isin(gap_dates)] = np.nan
non_d_italy_lpa_arrivals = non_d_italy_lpa_arrivals.interpolate(method='linear')

nd_italy_lpa_landed_path = os.path.join(BASE, "Non_domestic_italy_lpa_arrivals.csv")
non_d_italy_lpa_arrivals.rename('flight_count').to_csv(nd_italy_lpa_landed_path, index_label='date')

In [ ]:
nd_italy_lpa_landed_path = os.path.join(BASE, "Non_domestic_italy_lpa_arrivals.csv")
df_italy = pd.read_csv(nd_italy_lpa_landed_path, index_col='date', parse_dates=True)
series_italy = df_italy['flight_count']

In [ ]:
hermes_europe_countries = [
    'Austria', 'Belgium', 'Czech Republic', 'Denmark', 'Finland',
    'France', 'Germany', 'Greece', 'Hungary', 'Ireland', 'Italy',
    'Luxembourg', 'Netherlands', 'Norway', 'Poland', 'Portugal',
    'Spain', 'Sweden', 'Switzerland', 'United Kingdom'
]
cols = ['dof', 'ades_country_name', 'adep_country_name']
df_lpa_full = pd.read_parquet(FULL_LPA_FILE, columns=cols)
df_lpa_full['dof'] = pd.to_datetime(df_lpa_full['dof'])

mask = (
    (df_lpa_full['ades_country_name'].isin(hermes_europe_countries)) &
    (df_lpa_full['adep_country_name'] != df_lpa_full['ades_country_name'])
)
daily_hermes = df_lpa_full[mask].groupby('dof').size()

full_range = pd.date_range(start='2022-01-01', end='2025-12-31')
daily_hermes_arrivals = daily_hermes.reindex(full_range, fill_value=0).astype(float)

gap_dates = pd.to_datetime(possible_data_gaps)
daily_hermes_arrivals.loc[daily_hermes_arrivals.index.isin(gap_dates)] = np.nan
daily_hermes_arrivals = daily_hermes_arrivals.interpolate(method='linear')

nd_hermes_eu_lpa_landed_path = os.path.join(BASE, "Non_domestic_hermes_eu_lpa_arrivals.csv")
daily_hermes_arrivals.rename('flight_count').to_csv(nd_hermes_eu_lpa_landed_path, index_label='date')
print(f"Saved: {nd_hermes_eu_lpa_landed_path}")

In [ ]:
nd_hermes_eu_lpa_landed_path = os.path.join(BASE, "Non_domestic_hermes_eu_lpa_arrivals.csv")
df_hermes_eu  = pd.read_csv(nd_hermes_eu_lpa_landed_path, index_col='date', parse_dates=True)
series_hermes = df_hermes_eu['flight_count']

In [ ]:
quarters = pd.period_range(start='2022Q1', end='2025Q4', freq='Q')
q_non_d_italy_lpa_arrivals = (df_italy['flight_count']
                               .resample('QE')
                               .sum()
                               .values)

italy_arrivals_df = pd.DataFrame({
    'quarter': quarters.astype(str),
    'arrivals': q_non_d_italy_lpa_arrivals
})

q_non_domestic_lpa_arrivals_hermes_eu_locations = pd.DataFrame({
    'quarter': quarters.astype(str),
    'arrivals': daily_hermes_arrivals.resample('QE').sum().values
})
brunello_revenue = [
    24.207, 24.98,  28.9,   24.753,  # 2022 Q1-Q4
    27.559, 33.329, 36.312, 31.652,  # 2023 Q1-Q4
    34.947, 33.146, 42.207, 30.621,  # 2024 Q1-Q4
    38.775, 40.053, 41.472, 38.4     # 2025 Q1-Q4
]

brunello_df = pd.DataFrame({
    'quarter': quarters.astype(str),
    'revenue': brunello_revenue
})

hermes_revenue = [
     541,  635,  700,  724,
     664,  764,  814,  850,
     756,  894,  949,  994,
     857,  971, 1036, 1073
]

hermes_df = pd.DataFrame({
    'quarter': quarters.astype(str),
    'revenue': hermes_revenue
})

In [ ]:
def pct_change(series):
    values = [None]
    for i in range(1, len(series)):
        pct = (series[i] - series[i-1]) / series[i-1] * 100
        values.append(pct)
    return values

def plot_qoq(quarters, lpa_pct, brand_pct, lpa_label, brand_label, title):
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=quarters[1:], y=lpa_pct[1:],
        mode='lines+markers', name=lpa_label,
        line=dict(color='#F9E076', width=2),
        marker=dict(size=6, color='#F9E076', line=dict(color='black', width=0.8)),
        hovertemplate='<b>%{x}</b><br>LPA Change: %{y:.1f}%<extra></extra>'
    ))
    fig.add_trace(go.Scatter(
        x=quarters[1:], y=brand_pct[1:],
        mode='lines+markers', name=brand_label,
        line=dict(color='#B9B9B9', width=2),
        marker=dict(size=6, color='#B9B9B9', line=dict(color='black', width=0.8)),
        hovertemplate='<b>%{x}</b><br>Revenue Change: %{y:.1f}%<extra></extra>'
    ))
    fig.add_hline(y=0, line_dash='dash', line_color='black', line_width=0.8, opacity=0.4)
    fig.update_layout(
        font=dict(color='black'),
        title=dict(text=title, x=0.5, font=dict(size=14, color='black')),
        xaxis=dict(tickangle=-45, tickfont=dict(size=8), showgrid=True,
                   gridcolor='rgba(0,0,0,0.05)', linecolor='black', linewidth=1),
        yaxis=dict(title='<b>% Change vs Previous Quarter</b>', title_font=dict(size=10),
                   showgrid=True, gridcolor='rgba(0,0,0,0.05)',
                   linecolor='black', linewidth=1, ticksuffix='%'),
        paper_bgcolor='white', plot_bgcolor='white',
        height=450, width=800,
        margin=dict(t=100, b=120, l=60, r=60),
        legend=dict(orientation='h', yanchor='top', y=-0.2, xanchor='center', x=0.5, font=dict(size=10)),
        annotations=[dict(
            text="LPA* – Luxury Private Aviation",
            xref="paper", yref="paper",
            x=1, y=0, xanchor="right", yanchor="bottom",
            showarrow=False, font=dict(size=9, color="grey")
        )]
    )
    fig.show()
    return fig

In [ ]:
italy_arrivals_pct = pct_change(italy_arrivals_df['arrivals'].tolist())
brunello_pct       = pct_change(brunello_df['revenue'].tolist())
eu_arrivals_pct    = pct_change(q_non_domestic_lpa_arrivals_hermes_eu_locations['arrivals'].tolist())
hermes_pct         = pct_change(hermes_df['revenue'].tolist())

quarters_list = italy_arrivals_df['quarter'].tolist()

generated_fig = plot_qoq(
    quarters_list, italy_arrivals_pct, brunello_pct,
    lpa_label='Italy LPA Arrivals',
    brand_label='Brunello Cucinelli Italy Revenue',
    title='<b>Italy LPA* Arrivals vs Brunello Cucinelli Italy Revenue — QoQ % Change (2022–2025, Eurocontrol Area)</b><br>'
          '<sup>Source: Eurocontrol OPDI data & Brunello Cucinelli financial reports</sup>'
)
generated_fig.write_image("Figure 28. Italy LPA* Arrivals vs Brunello Cucinelli Italy Revenue — QoQ % Change (2022–2025, Eurocontrol Area).svg")

In [ ]:
generated_fig = plot_qoq(
    quarters_list, eu_arrivals_pct, hermes_pct,
    lpa_label='EU LPA Arrivals (Hermès Markets)',
    brand_label='Hermès Europe Revenue',
    title='<b>EU LPA* Arrivals vs Hermès Europe Revenue — QoQ % Change (2022–2025, Eurocontrol Area)</b><br>'
          '<sup>Source: Eurocontrol OPDI data & Hermès financial reports</sup>'
)
generated_fig.write_image("Figure 29. EU LPA* Arrivals vs Hermès Europe Revenue — QoQ % Change (2022–2025, Eurocontrol Area).svg")

#5. Forecasting Non-Domestic LPA Arrivals To France


##5.1. Data Preparation

###5.1.1 Time Series Data Creation and Missing Data Interpolation

For forecasting, I will create the non-domestic arrivals to France and use linear interpolation for dates that were source gaps.

In [ ]:
FULL_LPA_FILE = os.path.join(BASE, "full_lpa_eurocontrol_flights_2022_2025.parquet")

cols = ['dof', 'ades_country_name', 'adep_country_name']
df_lpa_full = pd.read_parquet(FULL_LPA_FILE, columns=cols)
df_lpa_full['dof'] = pd.to_datetime(df_lpa_full['dof'])

mask = (
    (df_lpa_full['ades_country_name'] == 'France') &
    (df_lpa_full['adep_country_name'] != 'France')
)
daily_non_d = df_lpa_full[mask].groupby('dof').size()

full_range = pd.date_range(start='2022-01-01', end='2025-12-31')
non_d_france_lpa_arrivals = daily_non_d.reindex(full_range, fill_value=0).astype(float)

gap_dates = pd.to_datetime(possible_data_gaps)
non_d_france_lpa_arrivals.loc[non_d_france_lpa_arrivals.index.isin(gap_dates)] = np.nan
non_d_france_lpa_arrivals = non_d_france_lpa_arrivals.interpolate(method='linear')

nd_france_lpa_landed_path = os.path.join(BASE, "Non_domestic_france_lpa_arrivals.csv")
non_d_france_lpa_arrivals.rename('flight_count').to_csv(nd_france_lpa_landed_path, index_label='date')

###5.1.2. Data Load

In [ ]:
nd_france_lpa_landed_path = os.path.join(BASE, "Non_domestic_france_lpa_arrivals.csv")
df     = pd.read_csv(nd_france_lpa_landed_path, index_col='date', parse_dates=True)
series = df['flight_count']
base_df = series.reset_index()
base_df.columns = ['ds', 'y']

print(f"Series shape:  {series.shape}")
print(f"Date range:    {series.index[0].date()} → {series.index[-1].date()}")
print(f"Nulls:         {series.isnull().sum()}")
print(f"base_df shape: {base_df.shape}")
print(base_df.head())

###5.1.2. Train, Validation, Test Split

In [ ]:
n         = len(base_df)
train_end = int(n * 0.70)
valid_end = int(n * 0.85)

train_df = base_df.iloc[:train_end].reset_index(drop=True)
valid_df = base_df.iloc[train_end:valid_end].reset_index(drop=True)
test_df  = base_df.iloc[valid_end:].reset_index(drop=True)   # final hold-out

train_valid_df = pd.concat([train_df, valid_df], ignore_index=True)
full_df        = pd.concat([train_df, valid_df, test_df], ignore_index=True)

initial_rows = len(train_df)

print(f"Total rows:   {n}")
print(f"Train:        {len(train_df)} rows  {train_df['ds'].iloc[0].date()} → {train_df['ds'].iloc[-1].date()}")
print(f"Validation:   {len(valid_df)} rows  {valid_df['ds'].iloc[0].date()} → {valid_df['ds'].iloc[-1].date()}")
print(f"Test:         {len(test_df)} rows  {test_df['ds'].iloc[0].date()} → {test_df['ds'].iloc[-1].date()}")

##5.2. Helper Codes

###5.2.1. Model Helper codes

In [ ]:
HORIZON = 21
STEP    = 9

def mae(a, p):  return np.abs(a - p).mean()
def rmse(a, p): return np.sqrt(((a - p) ** 2).mean())
def mape(a, p): return (np.abs((a - p) / np.where(a == 0, np.nan, a)) * 100).mean()

def print_metrics(label, actual, pred):
    return {
        'Model': label,
        'MAE':   round(mae(actual, pred), 2),
        'RMSE':  round(rmse(actual, pred), 2),
        'MAPE':  round(mape(actual, pred), 2)
    }

def walk_forward(predict_fn, df, initial_rows, horizon=HORIZON, step=STEP):
    actuals, preds, dates, horizon_idx = [], [], [], []
    for start in range(initial_rows, len(df) - horizon, step):
        wf_train = df.iloc[:start]
        wf_test  = df.iloc[start:start + horizon]
        pred     = predict_fn(wf_train, wf_test)
        min_len  = min(len(wf_test), len(pred))
        actuals.extend(wf_test['y'].values[:min_len])
        preds.extend(pred[:min_len])
        dates.extend(wf_test['ds'].values[:min_len])
        horizon_idx.extend(range(1, min_len + 1))
    return (np.array(actuals), np.array(preds),
            pd.to_datetime(dates), np.array(horizon_idx))

###I manually added upper window as event durations are diverse.
def make_holidays(name, starts, upper_window, lower_window=-3):
    return pd.DataFrame([
        {'holiday': name, 'ds': pd.Timestamp(s), 'lower_window': lower_window, 'upper_window': upper_window}
        for s in starts
    ])
    return pd.DataFrame(rows)

# Category groupings follow McCall et al. (2025): fashion/luxury, film/media, sport, yachting, trade.

events_raw = [
    ('Paris Haute Couture Week',      'fashion_luxury',  ['2022-01-24','2022-07-04','2023-01-23','2023-07-03','2024-01-22','2024-06-25','2025-01-27','2025-07-07'],   5 ),
    ('Paris Womenswear Fashion Week', 'fashion_luxury',  ['2022-02-28','2022-09-22','2023-02-27','2023-09-25','2024-02-26','2024-09-23','2025-02-25','2025-09-29'],   9 ),
    ('Paris Menswear Fashion Week',   'fashion_luxury',  ['2022-01-18','2022-06-21','2023-01-17','2023-06-20','2024-01-16','2024-06-18','2025-01-21','2025-06-24'],   6 ),
    ('Art Basel',                     'fashion_luxury',  ['2022-10-19','2023-10-20','2024-10-03','2025-10-18'],                                                        5 ),
    ('Maison et Objet',               'trade_fair',      ['2022-03-24','2022-09-08','2023-01-19','2023-09-07','2024-01-18','2024-09-05','2025-01-16','2025-09-04'],   5 ),
    ('Cannes Film Festival',          'film_media',      ['2022-05-17','2023-05-16','2024-05-14','2025-05-13'],                                                        12),
    ('Monaco Grand Prix',             'sport',           ['2022-05-27','2023-05-26','2024-05-23','2025-05-22'],                                                        3 ),
    ('Roland Garros',                 'sport',           ['2022-05-22','2023-05-28','2024-05-26','2025-05-25'],                                                        16),
    ('MIPIM',                         'trade_fair',      ['2022-03-15','2023-03-14','2024-03-12','2025-03-11'],                                                        4 ),
    ('Cannes Lions',                  'film_media',      ['2022-06-20','2023-06-19','2024-06-17','2025-06-16'],                                                        5 ),
    ('ILTM Cannes',                   'trade_fair',      ['2022-12-05','2023-12-04','2024-12-02','2025-12-01'],                                                        4 ),
    ('TFWA World Exhibition',         'trade_fair',      ['2022-10-02','2023-10-02','2024-09-29','2025-09-28'],                                                        5 ),
    ('Monaco Yacht Show',             'yachting',        ['2022-09-28','2023-09-27','2024-09-25','2025-09-24'],                                                        4 ),
    ('Cannes Yachting Festival',      'yachting',        ['2022-09-06','2023-09-12','2024-09-10','2025-09-09'],                                                        6 ),
    ('Paris Air Show',                'trade_fair',      ['2023-06-19','2025-06-16'],                                                                                  7 ),
    ('Paris Summer Olympics',         'sport',           ['2024-07-26'],                                                                                               17),
    ('Tour de France',                'sport',           ['2022-07-01','2023-07-01','2024-06-29','2025-07-05'],                                                        24),
]

holidays     = pd.concat([make_holidays(name, starts, uw) for name, _, starts, uw in events_raw], ignore_index=True)
holidays_cat = pd.concat([make_holidays(cat,  starts, uw) for _, cat, starts, uw in events_raw], ignore_index=True)

events = [(name, starts, uw) for name, _, starts, uw in events_raw]

###5.2.2. Visualization Helper Codes

In [ ]:
color_map = {}

def plot_walk_forward_by_cutoff(actuals, preds, dates, horizon_idx, title, color='crimson', n_cutoffs=6):
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=full_df[full_df['ds'] >= dates[0]]['ds'],
        y=full_df[full_df['ds'] >= dates[0]]['y'],
        name='Actual', line=dict(color='black', width=1.5)
    ))
    cutoff_starts = np.where(horizon_idx == 1)[0]
    indices  = np.linspace(0, len(cutoff_starts) - 1, n_cutoffs, dtype=int)
    selected = cutoff_starts[indices]
    for i, cs in enumerate(selected):
        ce = cs + HORIZON
        fig.add_trace(go.Scatter(
            x=dates[cs:ce], y=preds[cs:ce],
            name=f'Cutoff {i+1} ({pd.Timestamp(dates[cs]).date()})',
            line=dict(width=1.2, dash='dash'),
            opacity=0.7
        ))
    fig.update_layout(
        title=f'{title}<br><sup>Each dashed line = one 21-day forecast from its cutoff date</sup>',
        xaxis_title='Date', yaxis_title='Daily Arrivals',
        xaxis_range=[dates[0], dates[cutoff_starts[-1] + HORIZON - 1]],
        hovermode='x unified', template='plotly_white', height=500,
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
    )
    fig.show()

def plot_by_horizon(actuals, preds, horizon_idx, title, horizons=(1, 5, 14, 21)):
    rows = []
    for h in horizons:
        mask = (horizon_idx == h)
        if mask.sum() > 0:
            rows.append({'Horizon': f'{h}-ahead', 'MAE': mae(actuals[mask], preds[mask])})
    hdf = pd.DataFrame(rows)
    fig = go.Figure(go.Bar(x=hdf['Horizon'], y=hdf['MAE'], marker_color=color_map.get(title, '#F9E076')))
    fig.update_layout(title=f'{title} — MAE by Forecast Horizon',
                      xaxis_title='Horizon', yaxis_title='MAE',
                      template='plotly_white', height=350)
    fig.show()
    return hdf

def plot_residuals(actuals, preds, dates, title):
    resid_df = pd.DataFrame({'ds': dates, 'residual': actuals - preds})
    resid_df = resid_df.groupby('ds')['residual'].mean().reset_index()
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=resid_df['ds'], y=resid_df['residual'],
        mode='markers', marker=dict(color='black', size=4, opacity=0.7)
    ))
    fig.add_hline(y=0, line_dash='dash', line_color='gray')
    fig.update_layout(
        title=title, xaxis_title='Date', yaxis_title='Residual',
        template='plotly_white', height=300, showlegend=False
    )
    fig.show()
    print(f"Mean: {resid_df['residual'].mean():.2f}  Std: {resid_df['residual'].std():.2f}  Min: {resid_df['residual'].min():.2f}  Max: {resid_df['residual'].max():.2f}")

def plot_prophet_components(fitted_model, train_df, title='Prophet — Learned Components'):
    forecast  = fitted_model.predict(train_df[['ds']])
    day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
    one_week  = forecast.iloc[:7]
    sorted_pairs = sorted(
        zip(one_week['ds'].dt.day_name().tolist(), one_week['weekly'].tolist()),
        key=lambda x: day_order.index(x[0])
    )
    week_days_sorted, week_vals_sorted = zip(*sorted_pairs)
    fig = make_subplots(rows=3, cols=1,
        subplot_titles=['Trend', 'Weekly Seasonality', 'Yearly Seasonality'],
        vertical_spacing=0.15, row_heights=[0.4, 0.3, 0.3])
    fig.add_trace(go.Scatter(
        x=forecast['ds'], y=forecast['trend'],
        mode='lines', line=dict(color='#F9E076', width=1.2)
    ), row=1, col=1)
    fig.add_trace(go.Scatter(
        x=list(week_days_sorted), y=list(week_vals_sorted),
        mode='lines+markers', line=dict(color='darkorange', width=1.2), marker=dict(size=6)
    ), row=2, col=1)
    fig.add_trace(go.Scatter(
        x=forecast['ds'], y=forecast['yearly'],
        mode='lines', line=dict(color='grey', width=1.2)
    ), row=3, col=1)
    fig.update_layout(
        title=title, height=500, template='plotly_white',
        showlegend=False, hovermode='x unified'
    )
    fig.update_yaxes(range=[0, forecast['trend'].max() * 1.1], row=1, col=1)
    fig.show()

##5.3. Important Event Coverage Check

It is interesting to have a look at the highest arrival dates and whether they occured during the events we have added or not, this will help confirm that haven't missed major events - however, I admit that there are many more still events not added, but for time constraint reasons, the list contains important event sample.

In [ ]:
print(holidays.head(10))
print(f"Total holiday rows (individual): {len(holidays)}")
print(f"Total holiday rows (category):   {len(holidays_cat)}")

In [ ]:
cols = ['dof', 'ades_country_name', 'adep_country_name']
df_all = pd.read_parquet(FULL_LPA_FILE, columns=cols)
df_all['dof'] = pd.to_datetime(df_all['dof'])
mask_france = (df_all['ades_country_name'] == 'France') & (df_all['adep_country_name'] != 'France')
daily = df_all[mask_france].groupby('dof').size().rename('arrivals').reset_index()
daily.columns = ['ds', 'arrivals']

event_dates_list = []
for event_name, _, starts, lw, uw in events_raw:
    for start_str in starts:
        start        = pd.Timestamp(start_str)
        window_start = start + pd.Timedelta(days=lw)
        window_end   = start + pd.Timedelta(days=uw)
        for d in pd.date_range(window_start, window_end):
            event_dates_list.append({'ds': d, 'event': event_name})

event_lookup = (pd.DataFrame(event_dates_list)
    .groupby('ds')['event']
    .apply(lambda x: ', '.join(x.unique()))
    .reset_index())

daily_merged = daily.merge(event_lookup, on='ds', how='left')
daily_merged['event'] = daily_merged['event'].fillna('No added event')

top25 = daily_merged.nlargest(25, 'arrivals').reset_index(drop=True)
top25['ds'] = top25['ds'].dt.date

fig = go.Figure(data=[go.Table(
    columnwidth=[30, 70, 50, 300],
    header=dict(
        values=['#', 'Date', 'Arrivals', 'Events'],
        fill_color='#F4B942',
        font=dict(color='white', size=12),
        align='center',
        height=35
    ),
    cells=dict(
        values=[
            list(range(1, 26)),
            top25['ds'],
            top25['arrivals'],
            top25['event']
        ],
        fill_color=[['#f0f4fa' if i % 2 == 0 else 'white' for i in range(25)]],
        font=dict(color='#1a1a1a', size=11),
        align=['center', 'center', 'center', 'left'],
        height=30
    )
)])
fig.update_layout(
    title='Top 25 Days: LPA Flights Landed In France',
    template='plotly_white',
    height=950,
    width=1100,
    margin=dict(l=20, r=20, t=60, b=20)
)
fig.show()

It looks coverage of important relevant events nearly perfect.

##5.3 Baseline Model

In [ ]:
def naive_fn(train_df, test_df):
    return np.full(len(test_df), train_df['y'].iloc[-1])

def mean_fn(train_df, test_df):
    return np.full(len(test_df), train_df['y'].mean())

def snaive_fn(train_df, test_df):
    preds = []
    for ds in test_df['ds']:
        same_day_last_year = ds - pd.Timedelta(days=365)
        match = train_df[train_df['ds'] == same_day_last_year]['y']
        preds.append(match.values[0] if len(match) > 0 else np.nan)
    return np.array(preds)

act_naive,  pred_naive,  dates_naive,  hi_naive  = walk_forward(naive_fn,  train_valid_df, initial_rows)
act_mean,   pred_mean,   dates_mean,   hi_mean   = walk_forward(mean_fn,   train_valid_df, initial_rows)
act_snaive, pred_snaive, dates_snaive, hi_snaive = walk_forward(snaive_fn, train_valid_df, initial_rows)

mask = ~np.isnan(pred_snaive)

results = pd.DataFrame([
    print_metrics('Naive',                 act_naive,        pred_naive),
    print_metrics('Mean',                  act_mean,         pred_mean),
    print_metrics('Seasonal Naive (365d)', act_snaive[mask], pred_snaive[mask]),
])
results = results.set_index('Model')
print(results.to_string())

Baselines models perform quite good, especially seasonal naive. This is a strong seasonality indicator. And also the stability of LPA traffic.

##5.3. Prophet

Prophet is well-known for handling seasonal data well. Thus, as proved by seasonal naive model, it's a good model selection for this case.

###5.3.1. Best Parameter Search

- After trying default and default multiplicative, multiplicative worked better.
- Tuning reasons: to increase trend flexibility (changepoint_prior_scale=0.2) to better capture sharp demand shifts, extend changepoint detection window to 90% of training data to avoid missing recent structural changes, and increased candidate changepoints to 60 to give the model more granular trend-fitting opportunities.
- Tested difference between prophet with events additive vs multiplicative in suspicion that it won't learn much from events from multiplicative mode. However, evaluation metrics still performed better for multiplicative.
- tested with holidays_cat to see if it will predict better when events are combine, but it did better with just individual holidays and tuned settings mentioned.

In [ ]:
def make_prophet_fn(params):
    def _fn(train_df, test_df):
        m = Prophet(**params)
        m.fit(train_df)
        return m.predict(test_df[['ds']])['yhat'].values
    return _fn

p0_params  = {}
p1_params  = dict(seasonality_mode='multiplicative', daily_seasonality=False)
p2_params  = dict(seasonality_mode='multiplicative', daily_seasonality=False, changepoint_prior_scale=0.2,
                  changepoint_range=0.9, n_changepoints=60)
p3_params  = dict(seasonality_mode='multiplicative',daily_seasonality=False, holidays=holidays,
                  changepoint_prior_scale=0.2, changepoint_range=0.9, n_changepoints=60)
p4_params  = dict(seasonality_mode='multiplicative', daily_seasonality=False, holidays=holidays_cat,
                  holidays_prior_scale=0.05, changepoint_prior_scale=0.2,
                  changepoint_range=0.9, n_changepoints=60)
p5_params = dict(seasonality_mode='additive', daily_seasonality=False, holidays=holidays,
                 changepoint_prior_scale=0.2, changepoint_range=0.9, n_changepoints=60)


act_p0, pred_p0, dates_p0, hi_p0 = walk_forward(make_prophet_fn(p0_params), train_valid_df, initial_rows)
act_p1, pred_p1, dates_p1, hi_p1 = walk_forward(make_prophet_fn(p1_params), train_valid_df, initial_rows)
act_p2, pred_p2, dates_p2, hi_p2 = walk_forward(make_prophet_fn(p2_params), train_valid_df, initial_rows)
act_p3, pred_p3, dates_p3, hi_p3 = walk_forward(make_prophet_fn(p3_params), train_valid_df, initial_rows)
act_p4, pred_p4, dates_p4, hi_p4 = walk_forward(make_prophet_fn(p4_params), train_valid_df, initial_rows)
act_p5, pred_p5, dates_p5, hi_p5 = walk_forward(make_prophet_fn(p5_params), train_valid_df, initial_rows)


results = pd.DataFrame([
    print_metrics('Prophet Default',                                  act_p0, pred_p0),
    print_metrics('Prophet Multiplicative',                           act_p1, pred_p1),
    print_metrics('Prophet Tuned Multiplicative',                     act_p2, pred_p2),
    print_metrics('Prophet With Events Tuned Multiplicative',         act_p3, pred_p3),
    print_metrics('Prophet With Category Events Tuned Multiplicative',act_p4, pred_p4),
    print_metrics('Prophet With Events Tuned Additive',               act_p5, pred_p5),
]).set_index('Model')
print(results.to_string())

###5.3.2. Assessing Best Prophet Parameter Model

In [ ]:
m_p3 = Prophet(**p3_params).fit(train_df)
plot_walk_forward_by_cutoff(act_p3, pred_p3, dates_p3, hi_p3,
    title='Prophet With Events — Walk-Forward per Cutoff', color='purple')

Predictions successfully capture the shapes and constanly intersect with actual values.

In [ ]:
plot_prophet_components(m_p3, train_df, 'Prophet with Events — Learned Components')

In [ ]:
plot_residuals(act_p3, pred_p3, dates_p3, 'Prophet With Events — Residuals')

The model successfully learned weekly and yearly seasonaility showing clear patterns. When looking at resudauls it may seem like seasonality patterns still remain, however when inspected closer, they are rather random spikes and downs, hence, very likely just noise. And mean of residuals is 0.16, it's reached very good result.

In [ ]:
plot_by_horizon(act_p3, pred_p3, hi_p3, 'Prophet with Events')

As expected, predictions become worse with further dates in the 21horizon prediction. Although interestingly, 5 day ahead is even slighty lower in error.

###5.3.3. Prophet Event Learnings

In [ ]:
forecast_train = m_p3.predict(train_df[['ds']])

holiday_cols = [c for c in forecast_train.columns
                if c not in ['ds','trend','additive_terms','additive_terms_lower','additive_terms_upper',
                             'weekly','weekly_lower','weekly_upper','yearly','yearly_lower','yearly_upper',
                             'multiplicative_terms','multiplicative_terms_lower','multiplicative_terms_upper',
                             'yhat','yhat_lower','yhat_upper','trend_lower','trend_upper']
                and not c.endswith('_lower') and not c.endswith('_upper')]

n_cols = 3
n_rows = math.ceil(len(holiday_cols) / n_cols)

fig = make_subplots(rows=n_rows, cols=n_cols,
    subplot_titles=holiday_cols,
    vertical_spacing=0.08,
    horizontal_spacing=0.05)

for i, col in enumerate(holiday_cols):
    row = i // n_cols + 1
    col_pos = i % n_cols + 1
    fig.add_trace(go.Scatter(
        x=forecast_train['ds'], y=forecast_train[col],
        mode='lines', line=dict(color='orange', width=1.2)
    ), row=row, col=col_pos)

fig.update_layout(
    title='Prophet with Events — Learned Holiday Components',
    height=250 * n_rows,
    template='plotly_white', showlegend=False
)
fig.show()

In [ ]:
m_p5 = Prophet(**p5_params).fit(train_df)

forecast_train = m_p5.predict(train_df[['ds']])

holiday_cols = [c for c in forecast_train.columns
                if c not in ['ds','trend','additive_terms','additive_terms_lower','additive_terms_upper',
                             'weekly','weekly_lower','weekly_upper','yearly','yearly_lower','yearly_upper',
                             'multiplicative_terms','multiplicative_terms_lower','multiplicative_terms_upper',
                             'yhat','yhat_lower','yhat_upper','trend_lower','trend_upper']
                and not c.endswith('_lower') and not c.endswith('_upper')]

n_cols = 3
n_rows = math.ceil(len(holiday_cols) / n_cols)

fig = make_subplots(rows=n_rows, cols=n_cols,
    subplot_titles=holiday_cols,
    vertical_spacing=0.08,
    horizontal_spacing=0.05)

for i, col in enumerate(holiday_cols):
    row = i // n_cols + 1
    col_pos = i % n_cols + 1
    fig.add_trace(go.Scatter(
        x=forecast_train['ds'], y=forecast_train[col],
        mode='lines', line=dict(color='orange', width=1.2)
    ), row=row, col=col_pos)

fig.update_layout(
    title='Prophet with Events — Learned Holiday Components',
    height=250 * n_rows,
    template='plotly_white', showlegend=False
)
fig.show()

While multiplicative mode was slighly better performing, when it comes to learning about events, additive is much clearer to see the impact in growth. Some of these not annual events (paris air show, ILTM Cannes) learned negative components due to absent event years it seems. Top events - Cannes Film Festival reached 150, Monaco Yacht Show and Summer Olympics reached 100 and Roland Garros reached 50, prophet susccesfully learned about their strong impact and proved its relationship with LPA arrivals.

##5.4. LightGBM

###5.4.1. Feature Engineering

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7))
plot_acf(series.values, lags=366, ax=axes[0], title='Autocorrelation (ACF) — statsmodels CI',
         color='orange', vlines_kwargs={'colors': 'orange'}, alpha=0.05)
plot_acf_poly = [child for child in axes[0].get_children() if isinstance(child, plt.Polygon)]
for poly in plot_acf_poly:
    poly.set_facecolor('#F9E076')

sm_plot_pacf(series.values, lags=60, ax=axes[1], title='Partial Autocorrelation (PACF) — statsmodels CI',
             color='orange', vlines_kwargs={'colors': 'orange'}, alpha=0.05)
plot_pacf_poly = [child for child in axes[1].get_children() if isinstance(child, plt.Polygon)]
for poly in plot_pacf_poly:
    poly.set_facecolor('#F9E076')

plt.tight_layout()
plt.show()

As already confirmed in Prophet, there are weekly, yearly, monthly, seasonalities. So I will add day of year, day of week, and I want to explicitly instead of weekend choose just is_saturday, and year.

In [ ]:
def make_lgb_features(df):
    df = df.copy()
    df['dayofyear']   = df['ds'].dt.dayofyear
    df['dayofweek']   = df['ds'].dt.dayofweek
    df['is_saturday'] = (df['dayofweek'] == 5).astype(int)
    df['month']       = df['ds'].dt.month
    df['year']        = df['ds'].dt.year

    for lag in range(21, 28):
        df[f'lag_{lag}'] = df['y'].shift(lag)
    for window in range(21, 28):
        df[f'rolling_mean_{window}'] = df['y'].shift(21).rolling(window).mean()
        df[f'rolling_std_{window}']  = df['y'].shift(21).rolling(window).std()

    categories = list(dict.fromkeys(cat for _, cat, *_ in events_raw))
    for cat in categories:
        df[f'cat_{cat}'] = 0

    for event_name, cat, starts, _, uw in events_raw:
        col = event_name.lower().replace(' ', '_').replace('(', '').replace(')', '').replace('&', 'and')
        df[col] = 0
        for start_str in starts:
            start        = pd.Timestamp(start_str)
            window_start = start + pd.Timedelta(days=-3)
            window_end   = start + pd.Timedelta(days=uw)
            mask         = (df['ds'] >= window_start) & (df['ds'] <= window_end)
            df.loc[mask, col]          = 1
            df.loc[mask, f'cat_{cat}'] = 1

    return df

lgb_df       = make_lgb_features(base_df.copy()).dropna().reset_index(drop=True)
feature_cols = [c for c in lgb_df.columns if c not in ['ds', 'y']]
print(f"Rows after dropna: {len(lgb_df)}")
print(f"Features:          {len(feature_cols)}")

###5.4.2. Best Parameter Search

- Tried default
- Tuned version : Slowed learning rate further (0.005) paired with more trees (500) for smoother convergence, shallower trees (num_leaves=16) and stricter leaf size (min_child_samples=20) to reduce overfitting on a small daily series, added row and feature subsampling (subsample=0.8, colsample_bytree=0.8) to introduce stochastic regularization, and explicit L1/L2 penalties (reg_alpha=0.1, reg_lambda=1.0) to suppress noise from the sparse binary event features.

In [ ]:
lgb_train_valid_df = lgb_df[lgb_df['ds'] <= train_valid_df['ds'].iloc[-1]].reset_index(drop=True)
lgb_initial_rows   = lgb_train_valid_df[lgb_train_valid_df['ds'] >= valid_df['ds'].iloc[0]].index[0]

def make_lgb_fn(params):
    def _fn(train_df, test_df):
        model = lgb.LGBMRegressor(**params)
        model.fit(train_df[feature_cols], train_df['y'])
        return model.predict(test_df[feature_cols])
    return _fn

l0_params = dict(random_state=42)
l1_params = dict(n_estimators=200, learning_rate=0.01, num_leaves=20,
                 min_child_samples=10, random_state=42)

act_l0, pred_l0, dates_l0, hi_l0 = walk_forward(make_lgb_fn(l0_params), lgb_train_valid_df, lgb_initial_rows)
act_l1, pred_l1, dates_l1, hi_l1 = walk_forward(make_lgb_fn(l1_params), lgb_train_valid_df, lgb_initial_rows)

results = pd.DataFrame([
    print_metrics('LightGBM Default',                act_l0,     pred_l0),
    print_metrics('LightGBM Tuned',                  act_l1,     pred_l1),
]).set_index('Model')
print(results.to_string())

It looks like default settings work better for this dataset. It didn't perform better than Prophet.

###5.4.3. Feature Importance and SHAP Analysis

In [ ]:
lgb_train_valid_df = lgb_df[lgb_df['ds'] <= train_valid_df['ds'].iloc[-1]].reset_index(drop=True)
lgb_initial_rows   = lgb_train_valid_df[lgb_train_valid_df['ds'] >= valid_df['ds'].iloc[0]].index[0]

lgb_model_l0 = lgb.LGBMRegressor(**l0_params)
lgb_model_l0.fit(lgb_train_valid_df[feature_cols], lgb_train_valid_df['y'])

importance_l0 = (pd.DataFrame({'feature': feature_cols, 'importance': lgb_model_l0.feature_importances_})
                   .sort_values('importance', ascending=True))

fig = go.Figure(go.Bar(
    x=importance_l0['importance'], y=importance_l0['feature'],
    orientation='h', marker_color='#F9E076'
))
fig.update_layout(
    title='LightGBM Default — Feature Importance',
    xaxis_title='Importance', template='plotly_white', height=700
)
fig.show()

- `dayofyear dominates` — the model leans heavily on position in the year, essentially learning seasonality directly
- Lag features and rolling features appear on top and `is_saturday` was a good feature engineering as day of the week didn't appear even here.
- Rolling means/stds cluster in the middle — useful but secondary to lags
- `cat_sport` and `cat_yachting` appear next, so category feature engineering was better than singular event.
- `roland_garros`, `monaco_yacht_show` Remaining events are very close to zero, so it failed to learn about them.

In [ ]:
event_cols = [c for c in feature_cols if c.startswith('cat_') or c in [
    e[0].lower().replace(' ', '_').replace('(', '').replace(')', '').replace('&', 'and')
    for e in events_raw
]]

calendar_lag_cols = [c for c in feature_cols if c not in event_cols]

In [ ]:
explainer   = shap.TreeExplainer(lgb_model_l0)
shap_values = explainer.shap_values(lgb_train_valid_df[feature_cols])
orange_yellow = LinearSegmentedColormap.from_list('orange_yellow', ['#F9E076', 'orange', 'red'])

In [ ]:
cal_idx = [feature_cols.index(c) for c in calendar_lag_cols]
shap.summary_plot(shap_values[:, cal_idx], lgb_train_valid_df[calendar_lag_cols],
                  max_display=25, show=False, cmap=orange_yellow)
plt.title('SHAP Summary — Calendar, Lag & Rolling Features')
plt.tight_layout()
plt.show()

`dayofyear` is the single most influential feature, with high values (summer/late year) pushing predictions up to +30 and low values (winter) down to -30, confirming the model has learned a strong seasonal cycle. Recent momentum features (`rolling_mean_21`, `lag_21`) rank second and third — high recent averages and same-day-3-weeks-ago values consistently lift predictions, suggesting the series has meaningful short-term autocorrelation at the ~3-week horizon. Saturdays show a sharp, consistent negative effect (`is_saturday=1` clusters at -20 to -30), confirming a structural weekend suppression in non-domestic arrivals. dayofweek adds further directional signal but partially overlaps with is_saturday, which is worth acknowledging as mild redundancy. The remaining lags (22–27) and rolling statistics contribute marginally and cluster near zero, suggesting the feature set could be pruned without significant loss.

In [ ]:
event_idx = [feature_cols.index(c) for c in event_cols]
shap.summary_plot(shap_values[:, event_idx], lgb_train_valid_df[event_cols],
                  max_display=10, show=False, cmap=orange_yellow)
plt.title('SHAP Summary — Event Features')
plt.tight_layout()
plt.show()

##5.5 Hybrid (Prophet + LightGBM)

Prophet handles trend and seasonality well but misses short-term patterns and sharp event spikes. LightGBM picks up whatever systematic error Prophet leaves behind. Each model covers the other's weakness.

In [ ]:
def hybrid_fn(train_df, test_df):
    m = Prophet(**p3_params)
    m.fit(train_df)
    prophet_insample = m.predict(train_df[['ds']])
    resid_df = pd.DataFrame({
        'ds':       train_df['ds'].values,
        'residual': train_df['y'].values - prophet_insample['yhat'].values
    })
    lgb_train = make_lgb_features(train_df.copy()).dropna()
    lgb_train = lgb_train.merge(resid_df, on='ds', how='inner')
    lgb_model = lgb.LGBMRegressor(**l0_params)
    lgb_model.fit(lgb_train[feature_cols], lgb_train['residual'])
    prophet_pred  = m.predict(test_df[['ds']])['yhat'].values
    lgb_test      = make_lgb_features(pd.concat([train_df, test_df], ignore_index=True)).dropna()
    lgb_test      = lgb_test[lgb_test['ds'].isin(test_df['ds'])]
    residual_pred = lgb_model.predict(lgb_test[feature_cols])
    min_len = min(len(prophet_pred), len(residual_pred))
    return prophet_pred[:min_len] + residual_pred[:min_len]

act_h0, pred_h0, dates_h0, hi_h0 = walk_forward(hybrid_fn, train_valid_df, initial_rows)

results = pd.DataFrame([
    print_metrics('Seasonal Naive (365d)',           act_snaive[mask], pred_snaive[mask]),
    print_metrics('Best Prophet',                  act_p3,     pred_p3),
    print_metrics('Best LightGBM',                act_l0,     pred_l0),
    print_metrics('Hybrid (Best Prophet(Residuals) + Best LightGBM)',     act_h0,     pred_h0),
]).set_index('Model')
print(results.to_string())


Unfortunately, it didn't work well, so final model is Best Prophet(p3)

##5.6. Final Validation

In [ ]:
def final_predict_fn(train_df, test_df):
    m = Prophet(**p3_params)
    m.fit(train_df)
    return m.predict(test_df[['ds']])['yhat'].values

act_test_wf, pred_test_wf, dates_test_wf, hi_test_wf = walk_forward(
    final_predict_fn, full_df, len(train_valid_df)
)

final_metrics = pd.DataFrame([
    print_metrics('★ Chosen Model — Final Test (Prophet with Events)', act_test_wf, pred_test_wf),
]).set_index('Model')
print(final_metrics.to_string())

plot_walk_forward_by_cutoff(act_test_wf, pred_test_wf, dates_test_wf, hi_test_wf,
    title='Chosen Model — Final Test Evaluation (Unseen Data)', color='purple')